# Phase portraits over the slope maps: where the closed loop goes, and where the setpoint sits

**Question.** The LayerNorm kink is a small region around (x, u) ~ (0, 0) where the surrogate's slopes are wrong, with red
(gain too large) and blue (wrong-sign) arms radiating from it. Does it matter which coloured regions the closed loop passes
through on its way, or only where it ends up? And how does the placement of the setpoint relative to the kink change that?

**Time-ordered phase plots.** Each trajectory (x_k, u_k) on the TRUE plant is drawn in order (open circle = start, dots at
k = 1, 2, 5, 10, 20, 50, cross = k = 199) over the surrogate's input-gain map dF/du / B (white = correct, red = too large,
blue = wrong sign, black contour = sign flip) or its state-gain error map. Also drawn: the true equilibria u = u_ss(x) (dotted),
the surrogate's own equilibria F(x, u) = x (dashed: where a controller trained on the surrogate thinks it can rest), the
setpoint (star), the surrogate's steady-state input for it (diamond) and the closed-form kink ellipse. Under each panel:
the tracking error over time and the input gain the surrogate reports along the true path (red band = wrong sign).
The axes are symmetric-log (linear within |x| <= 0.25, |u| <= 1) so the kink and the far setpoints fit in one view.

**Three controllers, all designed from a surrogate and run on the true plant** `x+ = 1.03 x + 0.01 u`, cost
`sum Q (x - r)^2 + R (u - u_ss(r))^2 + Q_f (x_N - r)^2` (Q = 5, R = 0.1, Q_f = 50, N = 200), |u| <= 10:
* **LQR designed at the setpoint**: the surrogate is linearised once at its own equilibrium for r. Only the slopes AT THE
  SETPOINT enter, so the colours along the path should not matter: the negative control.
* **LQR rescheduled every step**: relinearised at (x_k, u_{k-1}) each step. The slopes ALONG THE PATH enter directly.
* **DPC**: (a) the regulation controllers already trained by the DPC notebook (setpoint 0), and (b) new **tracking**
  controllers: the GRU gets (x - r, r) and is trained through the surrogate on 1000 pairs x0 ~ U[-3, 3], r ~ U[-2, 2],
  with the loss measured around the surrogate's own equilibrium input (DPC only knows its model). Reference = the same
  tracking controller trained through the true plant.

**Setpoints** r in {-2, -1, -0.5, -0.25, -0.1, 0, 0.1, 0.25, 0.5, 1, 2}, 9 starts x0 in [-3, 3] each, all 10 seeds and all
six surrogates. The kink centre of every LayerNorm surrogate sits within 0.1 of (0, 0) (the encoder bias starts at zero), so
r = 0 puts the setpoint inside the kink and |r| >= 1 puts it many kink widths away.

**What was found already from the saved surrogates (LQR, no training needed)** - rerun here, plus the tracking DPC:
1. The input gain an LQR designed at the setpoint inherits is off by ~ 0.45 / d, d = distance of the setpoint from the kink
   centre in kink half-widths (log-log slope -0.98): the 1/distance fall-off of the petals. Wrong signs occur out to ~ 4-8
   half-widths (the petal arms).
2. With an accurate setpoint gain (within 20 %), the fixed LQR fails on ~ 1 % of runs whatever the path crosses; the
   rescheduled LQR fails on 1 % -> 9 % -> 27 % -> 40 % as the path's share of wrong-sign steps grows (0, 0-2, 2-10, > 10 %).
   So the colours along the path matter exactly when the controller uses the slopes along the path.

**Needs** the output folder of the DPC notebook (`MyDrive/dpc_out` on Colab: it holds `surrogates/` and `controllers/`).
Nothing in it is overwritten; this notebook adds `phase_lqr.csv`, `track_results.csv`, `track_controllers/`,
`track_trajectories/` and `figs/figP*`. **GPU:** A100 with `DTYPE = "float64"` (as the DPC run). The LQR part and the
figures take a few minutes; tracking DPC trains 7 models x 10 seeds, about 30-45 min per repeat on an A100. Re-running
skips what is done. `QUICK = True` checks everything in ~5 min with a tiny training (results in a separate folder).

In [ ]:
import importlib.util, subprocess, sys
for pkg in ("jax", "optax"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import jax
print("jax", jax.__version__, "| devices:", jax.devices())

In [ ]:
%%writefile level1_core.py
"""level1_core.py - Level 1 of the LayerNorm-kink method.

Level 1 = MSE + lambda x (Jacobian loss with labels estimated from the same transitions), the
slope-check points drawn uniformly from the training inputs. No lens, no trust weights, no other
tricks beyond the schedules being compared.

Everything is identical to smc_core.py (same plant, data seeds, model, initialisation, optimiser,
label estimator, k selection, sampling of slope-check points), so the numbers are comparable with
the E1 runs. The only differences: no label-trust weights (Level 1 has none; on box data they are
all 1 anyway), the Jacobian loss can use J_x only, J_u only or both, and the closed-form kink-line
evaluation points are clipped into the box (fixed count, so nothing recompiles).
"""
import os
import csv
import time
import numpy as np
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from scipy.spatial import cKDTree

tmap = jax.tree_util.tree_map

# ------------------------------------------------------------------------------------------------ #
# 0. Problem setup (Level 0, inherited from the TMLR theory paper)                                 #
# ------------------------------------------------------------------------------------------------ #
XMAX, UMAX = 4.0, 10.0          # data box: x ~ U[-4, 4], u ~ U[-10, 10]
N_DATA, N_TEST = 20000, 4000    # training transitions, noiseless test points
H, WB, EPS = 16, 32, 1e-6       # encoder width, MLP width, LayerNorm epsilon
LR = 3e-3                       # Adam learning rate, full batch
LQR_Q, LQR_R = 5.0, 0.1         # LQR weights of the closed-loop screen
K_GRID = (40, 80, 160, 320, 640, 1280)


class Plant:
    """x+ = A x + B u + eps_nl (a_sin sin x + c_xu x u). eps_nl = 0 is the linear plant."""

    def __init__(self, A=1.03, B=0.01, eps_nl=0.0, a_sin=0.05, c_xu=0.002):
        self.A, self.B, self.eps = float(A), float(B), float(eps_nl)
        self.a_sin, self.c_xu = float(a_sin), float(c_xu)

    def f(self, x, u):
        return self.A * x + self.B * u + self.eps * (self.a_sin * np.sin(x) + self.c_xu * x * u)

    def Jx(self, x, u):
        return self.A + self.eps * (self.a_sin * np.cos(x) + self.c_xu * u)

    def Ju(self, x, u):
        return self.B + self.eps * self.c_xu * x + 0.0 * u

    def u_ss(self, r):
        r = np.asarray(r, float)
        den = self.B + self.eps * self.c_xu * r
        with np.errstate(divide="ignore", invalid="ignore"):
            return (r - self.A * r - self.eps * self.a_sin * np.sin(r)) / den

    def r_max(self):
        """Setpoints the actuator can hold with a 10 % margin."""
        return min(0.9 * self.B * UMAX / max(self.A - 1.0, 1e-6), XMAX)


def make_data(plant, sigma, data_seed, n=N_DATA):
    rng = np.random.default_rng(data_seed)
    x = rng.uniform(-XMAX, XMAX, n)
    u = rng.uniform(-UMAX, UMAX, n)
    y = plant.f(x, u) + sigma * rng.standard_normal(n)
    return dict(x=x, u=u, y=y)


def make_test(plant):
    return make_data(plant, 0.0, 12345, N_TEST)


# ------------------------------------------------------------------------------------------------ #
# 1. Jacobian labels from the transitions (local quadratic, k nearest in std-scaled distance)      #
# ------------------------------------------------------------------------------------------------ #
def kernel_weights(t, kernel="epan"):
    """t = d/R in [0,1]. 'epan': 1 - t^2 (current code). 'gauss': exp(-2 t^2) (the note's rule)."""
    t = np.clip(t, 0.0, 1.0)
    if kernel == "epan":
        w = 1.0 - t ** 2
    elif kernel == "gauss":
        w = np.exp(-2.0 * t ** 2)
    else:
        raise ValueError(kernel)
    return np.maximum(w, 1e-12)


def _design(tx, tu):
    one = np.ones_like(tx)
    return np.stack([one, tx, tu, 0.5 * tx ** 2, tx * tu, 0.5 * tu ** 2], -1)


class LabelEstimator:
    """Local-quadratic weighted least squares around each query (x0, u0).
    d_i = sqrt(((x_i-x0)/s_x)^2 + ((u_i-u0)/s_u)^2), k nearest, R = d_(k), w_i = kernel(d_i/R),
    t = offsets / (s R), J_x = beta_1/(s_x R), J_u = beta_2/(s_u R). s_x, s_u = data std.
    Also returns v = Var(label)/sigma^2 from the sandwich formula."""

    def __init__(self, x, u, y, k, kernel="epan", ridge=1e-8):
        self.x, self.u, self.y = (np.asarray(a, float) for a in (x, u, y))
        self.k, self.kernel, self.ridge = int(k), kernel, float(ridge)
        self.sx, self.su = float(self.x.std()), float(self.u.std())
        self.tree = cKDTree(np.column_stack([self.x / self.sx, self.u / self.su]))

    def __call__(self, xq, uq, want_var=True):
        xq, uq = np.atleast_1d(np.asarray(xq, float)), np.atleast_1d(np.asarray(uq, float))
        chunk = max(32, 400_000 // self.k)
        n = len(xq)
        out = {k: np.empty(n) for k in ("Jx", "Ju", "vx", "vu", "R")}
        P = np.eye(6)
        P[0, 0] = 0.0
        for s in range(0, n, chunk):
            xs, us = xq[s:s + chunk], uq[s:s + chunk]
            d, idx = self.tree.query(np.column_stack([xs / self.sx, us / self.su]), k=self.k)
            R = np.maximum(d[:, -1:], 1e-12)
            w = kernel_weights(d / R, self.kernel)
            tx = (self.x[idx] - xs[:, None]) / (self.sx * R)
            tu = (self.u[idx] - us[:, None]) / (self.su * R)
            Phi = _design(tx, tu)
            PhiW = Phi * w[..., None]
            G = np.einsum("mki,mkj->mij", PhiW, Phi) + self.ridge * P
            Ginv = np.linalg.inv(G)
            beta = np.einsum("mij,mj->mi", Ginv, np.einsum("mki,mk->mi", PhiW, self.y[idx]))
            Rs = R[:, 0]
            out["Jx"][s:s + chunk] = beta[:, 1] / (self.sx * Rs)
            out["Ju"][s:s + chunk] = beta[:, 2] / (self.su * Rs)
            out["R"][s:s + chunk] = Rs
            if want_var:
                Cov = Ginv @ np.einsum("mki,mkj->mij", PhiW * w[..., None], Phi) @ Ginv
                out["vx"][s:s + chunk] = Cov[:, 1, 1] / (self.sx * Rs) ** 2
                out["vu"][s:s + chunk] = Cov[:, 2, 2] / (self.su * Rs) ** 2
        return out

    def noise_var(self, n_probe=2000, seed=0):
        """Pooled residual variance of unweighted local quadratic fits: an estimate of sigma^2."""
        rng = np.random.default_rng(seed)
        sel = rng.choice(len(self.x), size=min(n_probe, len(self.x)), replace=False)
        d, idx = self.tree.query(np.column_stack([self.x[sel] / self.sx, self.u[sel] / self.su]), k=self.k)
        R = np.maximum(d[:, -1:], 1e-12)
        Phi = _design((self.x[idx] - self.x[sel, None]) / (self.sx * R), (self.u[idx] - self.u[sel, None]) / (self.su * R))
        Yn = self.y[idx]
        G = np.einsum("mki,mkj->mij", Phi, Phi) + 1e-12 * np.eye(6)
        beta = np.linalg.solve(G, np.einsum("mki,mk->mi", Phi, Yn)[..., None])[..., 0]
        res = Yn - np.einsum("mki,mi->mk", Phi, beta)
        return float((res ** 2).sum() / (res.size - 6 * len(sel)))


def select_k(x, u, y, ks=K_GRID, kernel="epan", n_query=2000, seed=0):
    """k minimising the estimated label MSE (bias^2 + variance) in loss units, with no true Jacobian:
    variance = sigma_hat^2 x sandwich factor; bias^2(k) ~ mean[(J(2k) - J(k))^2] - sigma_hat^2 (v_k - v_2k)
    (doubling trick: the slope bias grows ~ linearly in k in 2-D); score = Var(x) MSE_x + Var(u) MSE_u.
    Returns (k, table, sigma_hat^2)."""
    x, u, y = (np.asarray(a, float) for a in (x, u, y))
    rng = np.random.default_rng(seed)
    q = rng.choice(len(x), min(n_query, len(x)), replace=False)
    xq, uq = x[q], u[q]
    ks = sorted(set(int(k) for k in ks))
    allk = [k for k in sorted(set(ks + [2 * k for k in ks])) if k < len(x)]
    labs = {k: LabelEstimator(x, u, y, k, kernel)(xq, uq) for k in allk}
    s2 = LabelEstimator(x, u, y, ks[0], kernel).noise_var()
    rows = []
    for k in ks:
        if 2 * k not in labs:
            continue
        a, b = labs[k], labs[2 * k]
        row = dict(k=k)
        for J, v in (("Jx", "vx"), ("Ju", "vu")):
            var_k = s2 * np.mean(a[v])
            bias2 = max(np.mean((b[J] - a[J]) ** 2) - s2 * (np.mean(a[v]) - np.mean(b[v])), 0.0)
            row[f"bias2_{J}"], row[f"var_{J}"], row[f"mse_{J}"] = bias2, var_k, bias2 + var_k
        row["score"] = np.var(x) * row["mse_Jx"] + np.var(u) * row["mse_Ju"]
        rows.append(row)
    best = min(rows, key=lambda r: r["score"])["k"]
    return best, rows, s2


def get_labels(d, seed, sigma, kernel, cache_dir):
    """Labels at every training input, computed once per dataset and cached to disk."""
    fn = os.path.join(cache_dir, f"labels_seed{seed}_sigma{sigma:g}_{kernel}.npz")
    if os.path.exists(fn):
        z = np.load(fn)
        return {k: z[k] for k in z.files}
    t0 = time.time()
    k, table, s2 = select_k(d["x"], d["u"], d["y"], kernel=kernel)
    lab = LabelEstimator(d["x"], d["u"], d["y"], k, kernel)(d["x"], d["u"])
    out = dict(Jx=lab["Jx"], Ju=lab["Ju"], vx=lab["vx"], vu=lab["vu"], k=np.array(k), sigma_hat=np.array(np.sqrt(s2)),
               ktab_k=np.array([r["k"] for r in table]), ktab_score=np.array([r["score"] for r in table]),
               ktab_bias2_Ju=np.array([r["bias2_Ju"] for r in table]), ktab_var_Ju=np.array([r["var_Ju"] for r in table]),
               ktab_bias2_Jx=np.array([r["bias2_Jx"] for r in table]), ktab_var_Jx=np.array([r["var_Jx"] for r in table]),
               time_s=np.array(time.time() - t0))
    np.savez_compressed(fn, **out)
    return out


# ------------------------------------------------------------------------------------------------ #
# 2. The surrogate (identical to kink_core / smc_core)                                             #
# ------------------------------------------------------------------------------------------------ #
def _glorot(key, shape):
    lim = np.sqrt(6.0 / (shape[0] + shape[1]))
    return jax.random.uniform(key, shape, minval=-lim, maxval=lim, dtype=jnp.float64)


def init_params(seed):
    k = jax.random.split(jax.random.PRNGKey(int(seed)), 4)
    kb = jax.random.split(k[2], 8)
    return dict(E=_glorot(k[0], (2, H)), b=jnp.zeros(H), gamma=jnp.ones(H), beta=jnp.zeros(H),
                Wd=_glorot(k[1], (H, 1)), bd=jnp.zeros(1),
                br=dict(W1=_glorot(kb[0], (H, WB)), b1=jnp.zeros(WB), W2=_glorot(kb[1], (WB, H)), b2=jnp.zeros(H)))


def stack_trees(trees):
    return tmap(lambda *a: jnp.stack(a), *trees)


def unstack_tree(tree, i):
    return tmap(lambda a: a[i], tree)


def _norm(kind, h, s0):
    if kind == "none":
        return h
    r = h - jnp.mean(h, -1, keepdims=True)
    if kind == "layer":
        return r / jnp.sqrt(jnp.mean(r ** 2, -1, keepdims=True) + EPS)
    if kind == "static":
        return r / s0
    raise ValueError(kind)


def F_point(p, x, u, s0, norm):
    h = x * p["E"][0] + u * p["E"][1] + p["b"]
    v = _norm(norm, h, s0)
    a = jax.nn.gelu((p["gamma"] * v + p["beta"]) @ p["br"]["W1"] + p["br"]["b1"], approximate=False)
    return h @ p["Wd"][:, 0] + (a @ p["br"]["W2"] + p["br"]["b2"]) @ p["Wd"][:, 0] + p["bd"][0]


def frozen_sigma(p, x, u):
    """Scale of the 'static' normalisation: the LayerNorm scale at initialisation, averaged over the data."""
    h = np.asarray(x)[:, None] * np.asarray(p["E"][0]) + np.asarray(u)[:, None] * np.asarray(p["E"][1]) + np.asarray(p["b"])
    r = h - h.mean(-1, keepdims=True)
    return float(np.sqrt((r ** 2).mean(-1) + EPS).mean())


def _Fb(norm):
    return jax.vmap(lambda p, x, u, s0: F_point(p, x, u, s0, norm), in_axes=(None, 0, 0, None))


def _derivs(Fb, p, x, u, s0):
    jx = jax.jvp(lambda xx: Fb(p, xx, u, s0), (x,), (jnp.ones_like(x),))[1]
    ju = jax.jvp(lambda uu: Fb(p, x, uu, s0), (u,), (jnp.ones_like(u),))[1]
    return jx, ju


# ------------------------------------------------------------------------------------------------ #
# 3. Loss and trainer (vmapped over seeds)                                                         #
# ------------------------------------------------------------------------------------------------ #
def make_loss(norm, kind, M):
    """kind 'mse':  mean (F - y)^2
       kind 'sob':  mean (F - y)^2 + lam * mean_j [ wx s_x^2 (dF/dx - Jx_lab)^2 + wu s_u^2 (dF/du - Ju_lab)^2 ]
    over M slope-check points drawn uniformly (with replacement) from the training inputs each step.
    wx, wu in {0, 1} select J_x only / J_u only / both. s_x, s_u = data std (unit-invariant loss)."""
    Fb = _Fb(norm)

    def loss(p, key, D, wx, wu, lam):
        mse = jnp.mean((Fb(p, D["x"], D["u"], D["s0"]) - D["y"]) ** 2)
        if kind == "mse":
            return mse
        n = D["x"].shape[0]
        idx = jnp.clip(jnp.searchsorted(D["cdf"], jax.random.uniform(key, (M,), dtype=jnp.float64)), 0, n - 1)
        jx, ju = _derivs(Fb, p, D["x"][idx], D["u"][idx], D["s0"])
        reg = jnp.mean(wx * D["sx"] ** 2 * (jx - D["Jx_lab"][idx]) ** 2 + wu * D["su"] ** 2 * (ju - D["Ju_lab"][idx]) ** 2)
        return mse + lam * reg

    return loss


_TRAINERS = {}


def get_trainer(norm, kind, M, chunk):
    key = (norm, kind, M, chunk)
    if key in _TRAINERS:
        return _TRAINERS[key]
    loss = make_loss(norm, kind, M)
    b1, b2, ae = 0.9, 0.999, 1e-8

    def one_chunk(p, m, v, t0, key, D, wx, wu, lam):
        def st(c, i):
            p, m, v, key = c
            key, sub = jax.random.split(key)
            t = t0 + i + 1.0
            g = jax.grad(loss)(p, sub, D, wx, wu, lam)
            m = tmap(lambda a, b: b1 * a + (1 - b1) * b, m, g)
            v = tmap(lambda a, b: b2 * a + (1 - b2) * b * b, v, g)
            p = tmap(lambda a, mm, vv: a - LR * (mm / (1 - b1 ** t)) / (jnp.sqrt(vv / (1 - b2 ** t)) + ae), p, m, v)
            return (p, m, v, key), 0.0
        (p, m, v, key), _ = jax.lax.scan(st, (p, m, v, key), jnp.arange(chunk, dtype=jnp.float64))
        return p, m, v, key

    run = jax.jit(jax.vmap(one_chunk, in_axes=(0, 0, 0, None, 0, 0, None, None, None)))
    _TRAINERS[key] = run
    return run


_EVALS = {}


def get_evaluator(norm):
    """Batched over seeds: (P, X[S,n], U[S,n], s0[S]) -> F, dF/dx, dF/du, each [S, n]."""
    if norm in _EVALS:
        return _EVALS[norm]
    Fb = _Fb(norm)

    def one(p, x, u, s0):
        jx, ju = _derivs(Fb, p, x, u, s0)
        return Fb(p, x, u, s0), jx, ju

    ev = jax.jit(jax.vmap(one, in_axes=(0, 0, 0, 0)))
    _EVALS[norm] = ev
    return ev


# ------------------------------------------------------------------------------------------------ #
# 4. Kink geometry (closed form, from the encoder weights) and evaluation                          #
# ------------------------------------------------------------------------------------------------ #
_PM = np.eye(H) - np.ones((H, H)) / H


def kink_center(p):
    """z_min = argmin ||P(E^T z + b)|| and the kink half-widths r*_x, r*_u (Theorem U2 of the theory paper)."""
    E, b = np.asarray(p["E"], float), np.asarray(p["b"], float)
    A = _PM @ E.T
    c0 = _PM @ b
    z = -np.linalg.lstsq(A, c0, rcond=None)[0]
    r = A @ z + c0
    D = np.sqrt(r @ r + H * EPS)
    return z, D / np.linalg.norm(A[:, 0]), D / np.linalg.norm(A[:, 1])


def kink_line_points(p, n_lines=41, n_pts=61, tmax=8.0):
    """Points on the closed-form kink lines (x-sweeps at fixed u and u-sweeps at fixed x), each dense around
    its own closest-approach point. Clipped into the box so the count is fixed (2 n_lines n_pts)."""
    E, b = np.asarray(p["E"], float), np.asarray(p["b"], float)
    th = np.tan(np.linspace(-np.arctan(tmax), np.arctan(tmax), n_pts))
    out = []
    for sweep, fixed, lim in ((0, np.linspace(-UMAX, UMAX, n_lines), XMAX), (1, np.linspace(-XMAX, XMAX, n_lines), UMAX)):
        q = _PM @ E[sweep]
        nq2 = float(q @ q)
        C = fixed[:, None] * (_PM @ E[1 - sweep])[None, :] + (_PM @ b)[None, :]
        s_star = -(C @ q) / nq2
        cp = C + s_star[:, None] * q[None, :]
        rstar = np.sqrt((cp ** 2).sum(1) + H * EPS) / np.sqrt(nq2)
        s = np.clip(s_star[:, None] + rstar[:, None] * th[None, :], -lim, lim)
        f0 = np.repeat(fixed[:, None], n_pts, 1)
        xy = np.stack([s, f0], -1) if sweep == 0 else np.stack([f0, s], -1)
        out.append(xy.reshape(-1, 2))
    return np.concatenate(out, 0)


def grid(n):
    xg, ug = np.meshgrid(np.linspace(-XMAX, XMAX, n), np.linspace(-UMAX, UMAX, n))
    return xg.ravel(), ug.ravel()


def dare_K(a, b, Q=LQR_Q, R=LQR_R):
    a, b = np.asarray(a, float), np.asarray(b, float)
    b2 = b * b
    coef = R - Q * b2 - a * a * R
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.where(b2 > 1e-14, (-coef + np.sqrt(coef ** 2 + 4 * b2 * Q * R)) / (2 * b2), np.nan)
        return np.where(b2 > 1e-14, a * b * P / (R + b2 * P), 0.0)


def eq_line(plant, n=801):
    rm = plant.r_max()
    r = np.linspace(-rm, rm, n)
    us = plant.u_ss(r)
    ok = np.isfinite(us) & (np.abs(us) <= UMAX)
    return r[ok], us[ok]


def lqr_screen(a_h, b_h, plant, r, us):
    """LQR (Q, R) designed from the MODEL's local (A^, B^) at each setpoint on the equilibrium line, applied
    to the TRUE local plant. 'local' uses the model Jacobian at the setpoint; 'win' averages it over +-W."""
    a_t, b_t = plant.Jx(r, us), plant.Ju(r, us)
    pole = a_t - b_t * dare_K(a_h, b_h)
    pole_o = a_t - b_t * dare_K(a_t, b_t)
    W = 0.25 * plant.B * UMAX / max(plant.A - 1.0, 1e-6) / 3.33
    k = max(1, int(round(W / max(r[1] - r[0], 1e-12))))
    ker = np.ones(2 * k + 1)
    nrm = np.convolve(np.ones_like(a_h), ker, "same")
    a_w, b_w = np.convolve(a_h, ker, "same") / nrm, np.convolve(b_h, ker, "same") / nrm
    pole_w = a_t - b_t * dare_K(a_w, b_w)
    return dict(lqr_unstable=float(np.mean(np.abs(pole) >= 1.0)), lqr_unstable_win=float(np.mean(np.abs(pole_w) >= 1.0)),
                lqr_pole_err_max=float(np.max(np.abs(pole - pole_o))), eq_gain_min=float(np.min(b_h / b_t)),
                eq_gain_max=float(np.max(b_h / b_t)))


CENTRE_WIN = (0.5, 1.5)    # |x| <= 0.5, |u| <= 1.5: 1.9 % of the box around (0, 0), where the LayerNorm kink sits


def kink_tests(E, X, U, sx=XMAX / np.sqrt(3.0), su=UMAX / np.sqrt(3.0)):
    """Four measurable signatures that separate a localised kink from smooth approximation error, computed from the
    input-gain error E = (dF/du - B)/B on the evaluation grid (E: [S, n, n], rows = u, columns = x)."""
    E = np.asarray(E, float)
    A = np.abs(E)
    S = E.shape[0]
    rms = np.sqrt((E ** 2).reshape(S, -1).mean(1))
    peak = A.reshape(S, -1).max(1)
    win = (np.abs(X) <= CENTRE_WIN[0]) & (np.abs(U) <= CENTRE_WIN[1])
    centre = (E ** 2)[:, win].sum(1) / np.maximum((E ** 2).reshape(S, -1).sum(1), 1e-300)
    dx, du = X[0, 1] - X[0, 0], U[1, 0] - U[0, 0]
    gU, gX = np.gradient(E, du, dx, axis=(1, 2))
    steep = np.sqrt((gX * sx) ** 2 + (gU * su) ** 2).reshape(S, -1).max(1)
    k = A.reshape(S, -1).argmax(1)
    wx, wu = X.ravel()[k], U.ravel()[k]
    xmax, umax = np.abs(X).max(), np.abs(U).max()
    where = np.where((np.abs(wx) <= 2 * CENTRE_WIN[0]) & (np.abs(wu) <= 2 * CENTRE_WIN[1]), "centre",
                     np.where((np.abs(wx) >= 0.95 * xmax) | (np.abs(wu) >= 0.95 * umax), "edge", "interior"))
    return dict(crest=peak / np.maximum(rms, 1e-300), centre_energy=centre, steep=steep, worst_x=wx, worst_u=wu, worst_where=where)


def evaluate_batch(P, S0, norm, plant, test, ngrid, full=False):
    """Metrics for every seed of a stacked model P. Never touches training data."""
    ev = get_evaluator(norm)
    S = len(S0)
    s0 = jnp.asarray(S0)
    tile = lambda a: jnp.asarray(np.repeat(np.asarray(a)[None], S, 0))
    Ft, _, _ = ev(P, tile(test["x"]), tile(test["u"]), s0)
    tmse = np.mean((np.asarray(Ft) - test["y"][None]) ** 2, 1)
    xg, ug = grid(ngrid)
    _, jxg, jug = ev(P, tile(xg), tile(ug), s0)
    jxg, jug = np.asarray(jxg), np.asarray(jug)
    ex, eu = jxg - plant.Jx(xg, ug)[None], (jug - plant.Ju(xg, ug)[None]) / plant.B
    out = dict(rel_mse=tmse / np.var(test["y"]), Jx_rms=np.sqrt((ex ** 2).mean(1)), Ju_rms_rel=np.sqrt((eu ** 2).mean(1)),
               Jx_sup=np.abs(ex).max(1), Ju_sup_rel=np.abs(eu).max(1),
               flip_area=np.mean(np.sign(jug) != np.sign(plant.Ju(xg, ug))[None], 1))
    inner = (np.abs(xg) <= 0.95 * XMAX) & (np.abs(ug) <= 0.95 * UMAX)        # box without its outer 5 % (edge effects)
    out["flip_area_interior"] = np.mean((np.sign(jug) != np.sign(plant.Ju(xg, ug))[None])[:, inner], 1)
    if norm == "layer":
        KL = np.stack([kink_line_points(unstack_tree(P, i)) for i in range(S)])
        _, jxk, juk = ev(P, jnp.asarray(KL[..., 0]), jnp.asarray(KL[..., 1]), s0)
        ekx = np.abs(np.asarray(jxk) - plant.Jx(KL[..., 0], KL[..., 1])).max(1)
        eku = np.abs((np.asarray(juk) - plant.Ju(KL[..., 0], KL[..., 1])) / plant.B).max(1)
        out["Jx_sup"] = np.maximum(out["Jx_sup"], ekx)
        out["Ju_sup_rel"] = np.maximum(out["Ju_sup_rel"], eku)
    if not full:
        return out, None
    out["gain_bad_area"] = np.mean(np.abs(jug / plant.Ju(xg, ug)[None] - 1.0) > 1.0, 1)
    out.update(kink_tests(eu.reshape(S, ngrid, ngrid), xg.reshape(ngrid, ngrid), ug.reshape(ngrid, ngrid)))
    r, us = eq_line(plant)
    _, jxe, jue = ev(P, tile(r), tile(us), s0)
    jxe, jue = np.asarray(jxe), np.asarray(jue)
    scr = [lqr_screen(jxe[i], jue[i], plant, r, us) for i in range(S)]
    for key in scr[0]:
        out[key] = np.array([s[key] for s in scr])
    geo = [kink_center(unstack_tree(P, i)) for i in range(S)] if norm == "layer" else [(np.full(2, np.nan), np.nan, np.nan)] * S
    out["zmin_x"] = np.array([g[0][0] for g in geo])
    out["zmin_u"] = np.array([g[0][1] for g in geo])
    out["rstar_x"] = np.array([g[1] for g in geo])
    out["rstar_u"] = np.array([g[2] for g in geo])
    zoom = []
    for i in range(S):
        cx = np.clip(out["zmin_x"][i] if np.isfinite(out["zmin_x"][i]) else 0.0, -XMAX + 1.5, XMAX - 1.5)
        cu = np.clip(out["zmin_u"][i] if np.isfinite(out["zmin_u"][i]) else 0.0, -UMAX + 4.0, UMAX - 4.0)
        zx, zu = np.meshgrid(np.linspace(cx - 1.5, cx + 1.5, 121), np.linspace(cu - 4.0, cu + 4.0, 121))
        zoom.append((zx.ravel(), zu.ravel()))
    ZX, ZU = np.stack([z[0] for z in zoom]), np.stack([z[1] for z in zoom])
    _, _, juz = ev(P, jnp.asarray(ZX), jnp.asarray(ZU), s0)
    prof = dict(eq_r=r, eq_u=us, eq_Ju=jue, eq_Jx=jxe, eq_Ju_true=plant.Ju(r, us), eq_Jx_true=plant.Jx(r, us),
                map_x=xg.reshape(ngrid, ngrid), map_u=ug.reshape(ngrid, ngrid),
                map_Ju=jug.reshape(S, ngrid, ngrid).astype(np.float32), map_Ju_true=plant.Ju(xg, ug).reshape(ngrid, ngrid),
                zoom_x=ZX.reshape(S, 121, 121), zoom_u=ZU.reshape(S, 121, 121),
                zoom_Ju=np.asarray(juz).reshape(S, 121, 121).astype(np.float32))
    return out, prof


# ------------------------------------------------------------------------------------------------ #
# 6. The experiment: configurations, resumable I/O                                                 #
# ------------------------------------------------------------------------------------------------ #
LOSSES = {"J_x only": (1.0, 0.0), "J_u only": (0.0, 1.0), "J_x + J_u": (1.0, 1.0)}


def level1_configs(T=3000, T_extra=1000):
    """14 configurations. T = main training length, T_extra = the extra phase of the 4000-step schedules.
    two-stage = MSE for T steps, then MSE + Jacobian loss for T_extra steps (fresh Adam moments at the switch)."""
    C = [dict(name="MSE (3000)", group="MSE", loss="MSE", schedule="3000", norm="layer", labels=None,
              phases=[("mse", 0, 0, T)]),
         dict(name="MSE (3000+1000)", group="MSE", loss="MSE", schedule="3000+1000", norm="layer", labels=None,
              phases=[("mse", 0, 0, T), ("mse", 0, 0, T_extra)]),
         dict(name="static norm, MSE", group="kink-free", loss="MSE", schedule="3000", norm="static", labels=None,
              phases=[("mse", 0, 0, T)]),
         dict(name="no norm, MSE", group="kink-free", loss="MSE", schedule="3000", norm="none", labels=None,
              phases=[("mse", 0, 0, T)])]
    for lname, (wx, wu) in LOSSES.items():
        C += [dict(name=f"{lname} (3000)", group=lname, loss=lname, schedule="3000", norm="layer", labels="est",
                   phases=[("sob", wx, wu, T)]),
              dict(name=f"{lname} (4000)", group=lname, loss=lname, schedule="4000", norm="layer", labels="est",
                   phases=[("sob", wx, wu, T + T_extra)]),
              dict(name=f"{lname} (two-stage)", group=lname, loss=lname, schedule="two-stage", norm="layer", labels="est",
                   phases=[("mse", 0, 0, T), ("sob", wx, wu, T_extra)])]
    C.append(dict(name="J_x + J_u, TRUE labels (3000)", group="true labels", loss="J_x + J_u", schedule="3000",
                  norm="layer", labels="true", phases=[("sob", 1.0, 1.0, T)]))
    return C


def out_root(name="level1_out"):
    if os.environ.get("L1_OUT_ROOT"):
        return os.environ["L1_OUT_ROOT"]
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
    return os.path.join(base, name)


def append_csv(path, rows):
    if not rows:
        return
    new = not os.path.exists(path)
    cols = list(rows[0].keys())
    if not new:
        with open(path) as f:
            cols = next(csv.reader(f))
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerows(rows)


def safe(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")


In [ ]:
%%writefile dpc_core.py
"""dpc_core.py - Differentiable Predictive Control (DPC) THROUGH the Level 1 surrogates.

Pipeline (the user's DPC example, applied to the LayerNorm-kink study):
  1. Identify the plant: retrain the Level 1 surrogates F(x, u) exactly as in level1_core (same data, init, labels,
     draws), 10 seeds per model.
  2. Train a GRU controller by unrolling it through the FULL surrogate x_{k+1} = F(x_k, u_k) (no linear extraction,
     so the controller sees the kink), with the LQR-type loss and the horizon curriculum of the example.
  3. Test every controller on the TRUE plant x+ = 1.03 x + 0.01 u. Reference = the same controller trained through the
     true plant ("oracle"); the headline number is the true-plant cost gap.

Changes from the example, each needed for this study: rollout through the whole network (not W, V after LayerNorm);
input bounded to |u| <= 10 by u = 10 tanh(head) (the surrogates only saw |u| <= 10 and the plant's actuator is limited
there); plain JAX instead of flax.nnx so the 10 seeds train together (same GRU equations, optax.adamw); per seed every
model's controller starts from the same weights and trains on the same 1000 initial states (paired comparisons).
"""
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
import optax
import level1_core as L

tmap = jax.tree_util.tree_map

# ------------------------------------------------------------------------------------------------ #
# 0. Settings (the example's values)                                                               #
# ------------------------------------------------------------------------------------------------ #
CONFIG = dict(N=200, Q_x=5.0, R_u=0.1, Q_f=50.0)          # LQR-type loss and test horizon
CURRICULUM = [(5, 1000), (10, 2000), (20, 2000), (50, 1000), (100, 1000), (200, 2000)]   # (horizon, epochs)
HIDDEN = 64                                               # GRU width
CTRL_LR = 1e-3                                            # optax.adamw (default weight decay 1e-4, as in the example)
N_X0 = 1000                                               # training initial states per controller
X0_RANGE = 3.0                                            # x0 ~ U[-3, 3] (controllable range with |u| <= 10: |x| < 3.33)
U_BOUND = 10.0                                            # u = 10 tanh(head)
X0_EVAL = np.array(sorted({s * v for v in (0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0) for s in (-1, 1)}))
N_TEST = 1000                                             # "typical" test starts, x0 ~ U[-3, 3], same for every seed
DIVERGE = 10.0                                            # |x| above this anywhere = diverged
SETTLED = 0.05                                            # |x_N| above this at the end = not regulated
LQR_K = 10.29                                             # analytic LQR gain for (1.03, 0.01, Q=5, R=0.1)

# The models compared. 'surrogate' names a Level 1 configuration (level1_configs or wave_configs).
MODELS = [dict(name="true plant (oracle)", short="oracle", kind="true", group="oracle"),
          dict(name="LN, MSE (3000)", short="LN MSE", surrogate="MSE (3000)", group="MSE"),
          dict(name="LN, MSE (12000)", short="LN MSE\n12000", surrogate="MSE (12000)", group="MSE long"),
          dict(name="LN, J_x + J_u (3000)", short="LN slope\nfrom step 0", surrogate="J_x + J_u (3000)", group="J_x + J_u"),
          dict(name="LN, two-stage", short="LN\ntwo-stage", surrogate="J_x + J_u (two-stage)", group="two-stage"),
          dict(name="no norm, MSE", short="no\nnorm", surrogate="no norm, MSE", group="kink-free"),
          dict(name="static norm, MSE", short="static\nnorm", surrogate="static norm, MSE", group="kink-free")]

# Optimiser-side control (2026-10-23): the same LN MSE surrogate, but DPC aggregates per-start gradients with a bounded vote
# (robust_grad 'clip'). If this rescues the trapped seeds, the failure is the exploding per-trajectory gradient, and this is
# the generic "fix the optimiser" baseline. Not in MODELS by default: add it in the notebook (ROBUST_MODELS) for the A100 run.
ROBUST_MODELS = [dict(name="LN, MSE (3000), clipped grads", short="LN MSE\nclipped", surrogate="MSE (3000)", group="MSE robust",
                      aggregate="clip", clip=3.0),
                 dict(name="LN, MSE (3000), unit grads", short="LN MSE\nunit vote", surrogate="MSE (3000)", group="MSE robust",
                      aggregate="unit")]


def surrogate_configs(T=3000, T_extra=1000, T_long=12000):
    """name -> Level 1 configuration, with the step counts scaled for QUICK runs."""
    C = {c["name"]: c for c in L.level1_configs(T, T_extra)}
    long = dict(name="MSE (12000)", group="MSE", loss="MSE", schedule=str(T_long), norm="layer", labels=None,
                phases=[("mse", 0, 0, T_long)])
    C["MSE (12000)"] = long
    return C


def safe(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name).strip("_")


def out_root(name="dpc_out"):
    """DPC_OUT_ROOT (Colab: MyDrive/dpc_out) or a local folder. arch_dir() appends the controller architecture."""
    if os.environ.get("DPC_OUT_ROOT"):
        return os.environ["DPC_OUT_ROOT"]
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
    return os.path.join(base, name)


# ------------------------------------------------------------------------------------------------ #
# 1. Surrogates: retrain all seeds of one Level 1 configuration together, save the weights          #
# ------------------------------------------------------------------------------------------------ #
def _flat(tree, prefix=""):
    out = {}
    for k, v in tree.items():
        if isinstance(v, dict):
            out.update(_flat(v, prefix + k + "/"))
        else:
            out[prefix + k] = np.asarray(v)
    return out


def _unflat(d):
    out = {}
    for k, v in d.items():
        parts = k.split("/")
        cur = out
        for p in parts[:-1]:
            cur = cur.setdefault(p, {})
        cur[parts[-1]] = jnp.asarray(v)
    return out


def train_surrogates(cfg, seeds, plant, sigma, kernel, out_dir, label_dir=None, M=2048, chunk=250, lam=1.0):
    """Stacked weights P [S, ...] and static-norm scales S0 [S] for one Level 1 configuration, trained exactly as
    level1_core.run_config does (same data, init, labels, slope-check draws, Adam restart at each phase switch).
    Cached in out_dir/surrogates/<name>.npz; also returns the Level 1 metrics of the trained models."""
    os.makedirs(os.path.join(out_dir, "surrogates"), exist_ok=True)
    fn = os.path.join(out_dir, "surrogates", safe(cfg["name"]) + ".npz")
    if os.path.exists(fn):
        z = np.load(fn, allow_pickle=True)
        if list(z["seeds"]) == list(seeds):
            P = _unflat({k[2:]: z[k] for k in z.files if k.startswith("p:")})
            return P, z["S0"], {k[2:]: z[k] for k in z.files if k.startswith("m:")}
    label_dir = label_dir or out_dir
    t0 = time.time()
    datas, P0, S0, Jx_lab, Ju_lab = [], [], [], [], []
    for s in seeds:
        d = L.make_data(plant, sigma, 1000 + s)
        p = L.init_params(s)
        datas.append(d), P0.append(p)
        S0.append(L.frozen_sigma(p, d["x"], d["u"]) if cfg["norm"] == "static" else 1.0)
        if cfg.get("labels") == "est":
            lab = L.get_labels(d, s, sigma, kernel, label_dir)
            Jx_lab.append(lab["Jx"]), Ju_lab.append(lab["Ju"])
        elif cfg.get("labels") == "true":
            Jx_lab.append(plant.Jx(d["x"], d["u"])), Ju_lab.append(plant.Ju(d["x"], d["u"]))
        else:
            Jx_lab.append(np.zeros(L.N_DATA)), Ju_lab.append(np.zeros(L.N_DATA))
    D = tmap(jnp.asarray, dict(x=np.stack([d["x"] for d in datas]), u=np.stack([d["u"] for d in datas]),
                               y=np.stack([d["y"] for d in datas]), s0=np.array(S0),
                               sx=np.array([d["x"].std() for d in datas]), su=np.array([d["u"].std() for d in datas]),
                               cdf=np.tile(np.arange(1, L.N_DATA + 1) / L.N_DATA, (len(seeds), 1)),
                               Jx_lab=np.stack(Jx_lab), Ju_lab=np.stack(Ju_lab)))
    P = L.stack_trees(P0)
    KEYS = jnp.stack([jax.random.PRNGKey(10_000 + s) for s in seeds])
    lam = float(cfg.get("lam", lam))
    for kind, wx, wu, nsteps in cfg["phases"]:
        run = L.get_trainer(cfg["norm"], kind, M, chunk)
        Mo, Vo = tmap(jnp.zeros_like, P), tmap(jnp.zeros_like, P)
        for ci in range(max(1, nsteps // chunk)):
            P, Mo, Vo, KEYS = run(P, Mo, Vo, float(ci * chunk), KEYS, D, float(wx), float(wu), lam)
    jax.block_until_ready(P)
    met, _ = L.evaluate_batch(P, S0, cfg["norm"], plant, L.make_test(plant), 201, full=True)
    keep = ("rel_mse", "Ju_sup_rel", "Ju_rms_rel", "Jx_sup", "flip_area_interior", "lqr_unstable",
            "zmin_x", "zmin_u", "rstar_x", "rstar_u", "crest")
    met = {k: np.asarray(met[k], float) for k in keep if k in met}
    np.savez_compressed(fn, seeds=np.array(seeds), S0=np.array(S0), train_s=np.array(time.time() - t0),
                        **{"p:" + k: v for k, v in _flat(P).items()}, **{"m:" + k: v for k, v in met.items()})
    return P, np.array(S0), met


# ------------------------------------------------------------------------------------------------ #
# 2. Controllers. ARCH selects one; every controller has init(seed, d_x) -> c, step(c, h, x) -> (h, u), and a   #
#    hidden size (0 = memoryless: h is a dummy [B, 0] array, so rollout / phase code needs no special case).     #
#    'gru'    the example's StandaloneGRUController in plain JAX (hidden 64): the original study.               #
#    'mlp'    u = 10 tanh(MLP(x)), tanh hidden layers, memoryless: a plain policy u = pi(x), so phase plots show #
#             a function and K_eff = -d pi/dx is comparable to the LQR gain 10.29.                               #
#    'linear' u = 10 tanh((k x + b) / 10) ~ sat(-K x + c), 2 parameters per input: the whole DPC problem is a    #
#             2-D landscape that can be plotted exhaustively.                                                   #
# ------------------------------------------------------------------------------------------------ #
ARCH = "gru"                                              # default = the original study; set before training
MLP_WIDTH, MLP_DEPTH = 64, 2                              # 'mlp': hidden layers (tanh)
LIN_K0 = -5.0                                             # 'linear': initial gain u ~ LIN_K0 x (stabilising, half the LQR gain)
LIN_SCALE = 10.0                                          # 'linear' parameters are stored / LIN_SCALE so Adam's unit steps fit


def _linear_init(key, n_in, n_out, dtype):
    """flax.nnx.Linear default: lecun_normal kernel, zero bias."""
    return dict(W=jax.nn.initializers.lecun_normal()(key, (n_in, n_out), dtype), b=jnp.zeros(n_out, dtype))


def hidden_size(arch=None):
    return HIDDEN if (arch or ARCH) == "gru" else 0


def init_controller(seed, d_x=1, d_u=1, hidden=HIDDEN, dtype=jnp.float64, arch=None):
    """Controller weights from key 20000 + seed (run_keys passes seed + 1000 * repeat for repeats > 0).
    The same key gives the same GRU as before; the MLP and linear controllers draw from it too."""
    arch = arch or ARCH
    key = jax.random.PRNGKey(20_000 + int(seed))
    if arch == "gru":
        k = jax.random.split(key, 4)
        return dict(Wz=_linear_init(k[0], d_x + hidden, hidden, dtype), Wr=_linear_init(k[1], d_x + hidden, hidden, dtype),
                    Wh=_linear_init(k[2], d_x + hidden, hidden, dtype), head=_linear_init(k[3], hidden, d_u, dtype))
    if arch == "mlp":
        k = jax.random.split(key, MLP_DEPTH + 1)
        sizes = [d_x] + [MLP_WIDTH] * MLP_DEPTH + [d_u]
        return {f"L{i}": _linear_init(k[i], sizes[i], sizes[i + 1], dtype) for i in range(MLP_DEPTH + 1)}
    if arch == "linear":                                  # gain on the FIRST input (x, or x - r for tracking), 0 on the rest
        k = jnp.zeros((d_x, d_u), dtype).at[0, 0].set(LIN_K0 / LIN_SCALE)
        return dict(k=k, b=jnp.zeros(d_u, dtype))
    raise ValueError(arch)


def gru_step(c, h_prev, x):
    """h_new, u for a batch: x [B, d_x], h [B, hidden]. Same gates as the example; u = 10 tanh(head(h))."""
    lin = lambda l, a: a @ l["W"] + l["b"]
    hx = jnp.concatenate([x, h_prev], -1)
    z = jax.nn.sigmoid(lin(c["Wz"], hx))
    r = jax.nn.sigmoid(lin(c["Wr"], hx))
    h_tilde = jnp.tanh(lin(c["Wh"], jnp.concatenate([x, r * h_prev], -1)))
    h_new = (1 - z) * h_prev + z * h_tilde
    return h_new, U_BOUND * jnp.tanh(lin(c["head"], h_new))


def mlp_step(c, h, x):
    names = sorted(k for k in c if k.startswith("L"))      # L0 ... Ln
    a = x
    for nm in names[:-1]:
        a = jnp.tanh(a @ c[nm]["W"] + c[nm]["b"])
    l = c[names[-1]]
    return h, U_BOUND * jnp.tanh(a @ l["W"] + l["b"])


def linear_step(c, h, x):
    """u = 10 tanh((x K + B) / 10) with K = LIN_SCALE k, B = LIN_SCALE b: linear with gain K (u ~ K x + B) until it
    saturates at +-10. Stored parameters are k = K / 10, so an Adam step of 1e-3 moves the gain by 0.01."""
    return h, U_BOUND * jnp.tanh((x @ c["k"] + c["b"]) * LIN_SCALE / U_BOUND)


def ctrl_step(c, h, x):
    """Dispatch on the parameter tree, so a saved controller of any architecture runs with the same code."""
    if "Wz" in c:
        return gru_step(c, h, x)
    if "L0" in c:
        return mlp_step(c, h, x)
    return linear_step(c, h, x)


def policy_gain(c, x, r=None):
    """K_eff = -du/dx of a memoryless controller at x (scalar or array; r for tracking controllers)."""
    def u_of(xx):
        inp = jnp.array([xx]) if r is None else jnp.array([xx - r, r])
        return ctrl_step(c, jnp.zeros((1, 0)), inp[None])[1][0, 0]
    return -np.asarray(jax.vmap(jax.grad(u_of))(jnp.atleast_1d(jnp.asarray(x, float))))


# ------------------------------------------------------------------------------------------------ #
# 3. Dynamics the controller is trained through, and the DPC loss                                  #
# ------------------------------------------------------------------------------------------------ #
def make_dynamics(kind, norm=None, plant=None):
    """kind 'true': x+ = A x + B u. kind 'surrogate': x+ = F(x, u), the whole network (norm = layer/static/none)."""
    if kind == "true":
        A, B = float(plant.A), float(plant.B)
        return lambda p, s0, x, u: A * x + B * u
    Fb = jax.vmap(lambda p, x, u, s0: L.F_point(p, x, u, s0, norm), in_axes=(None, 0, 0, None))
    return lambda p, s0, x, u: Fb(p, x[:, 0], u[:, 0], s0)[:, None]


def rollout(c, dyn, p, s0, x0, N, hidden=None):
    """Closed loop of N steps through dyn. Returns xs [N+1, B, 1], us [N, B, 1]."""
    h0 = jnp.zeros((x0.shape[0], HIDDEN if "Wz" in c else 0), x0.dtype)

    def step(carry, _):
        h, x = carry
        h, u = ctrl_step(c, h, x)
        xn = dyn(p, s0, x, u)
        return (h, xn), (x, u)

    # Gradient checkpointing: keep only the carried (h, x) per step and recompute the step's internals in the backward
    # pass. Without it a full-size step (200 steps x 1000 starts x 10 seeds) needs ~13 GB for backpropagation.
    (h, xN), (xs, us) = jax.lax.scan(jax.checkpoint(step), (h0, x0), None, length=N)
    return jnp.concatenate([xs, xN[None]], 0), us


def dpc_loss(c, dyn, p, s0, x0, N, cfg=CONFIG):
    """The example's compute_lqr_loss: (sum_k mean(Q x_k^2 + R u_k^2) + mean(Q_f x_N^2)) / N."""
    xs, us = rollout(c, dyn, p, s0, x0, N)
    stage = jnp.sum(jnp.mean(cfg["Q_x"] * xs[:-1, :, 0] ** 2 + cfg["R_u"] * us[:, :, 0] ** 2, axis=1))
    return (stage + jnp.mean(cfg["Q_f"] * xs[-1, :, 0] ** 2)) / N


_DPC_STEPS = {}


def robust_grad(c, dyn, p, s0, x0, N, aggregate="mean", clip=1.0):
    """Loss and gradient over the batch of starts x0 [B, 1] with a choice of aggregation over starts.
    'mean'  : the ordinary full-batch gradient (gradient of the mean loss).
    'clip'  : per-start gradients, each clipped to norm <= clip, then averaged (a bounded vote per trajectory: one start
              whose rollout runs through a locally unstable region of the surrogate cannot dominate the batch).
    'unit'  : per-start gradients normalised to unit norm, then averaged (equal vote, no threshold).
    Per-start gradients cost one backward pass per start (vmap over starts), ~B x the memory of the mean gradient."""
    if aggregate == "mean":
        return jax.value_and_grad(dpc_loss)(c, dyn, p, s0, x0, N)
    one = lambda c, x: dpc_loss(c, dyn, p, s0, x[None], N)
    losses, G = jax.vmap(jax.value_and_grad(one), in_axes=(None, 0))(c, x0)          # [B], tree of [B, ...]
    nrm = jnp.sqrt(sum(jnp.sum(g.reshape(g.shape[0], -1) ** 2, 1) for g in jax.tree_util.tree_leaves(G)))   # [B]
    scale = jnp.minimum(1.0, clip / (nrm + 1e-30)) if aggregate == "clip" else 1.0 / (nrm + 1e-30)
    G = tmap(lambda g: jnp.mean(g * scale.reshape((-1,) + (1,) * (g.ndim - 1)), 0), G)
    return jnp.mean(losses), G


def get_dpc_trainer(key, dyn, N, n_epochs, opt, aggregate="mean", clip=1.0):
    """n_epochs full-batch AdamW steps at horizon N, vmapped over seeds. Returns (C, O, losses[S, n_epochs]).
    aggregate: how per-start gradients are combined (see robust_grad)."""
    k = (key, N, n_epochs, aggregate, clip)
    if k in _DPC_STEPS:
        return _DPC_STEPS[k]

    def one(c, o, p, s0, x0):
        def st(carry, _):
            c, o = carry
            loss, g = robust_grad(c, dyn, p, s0, x0, N, aggregate, clip)
            upd, o = opt.update(g, o, c)
            return (optax.apply_updates(c, upd), o), loss
        (c, o), losses = jax.lax.scan(st, (c, o), None, length=n_epochs)
        return c, o, losses

    f = jax.jit(jax.vmap(one, in_axes=(0, 0, 0, 0, 0)))
    _DPC_STEPS[k] = f
    return f


def run_keys(seed, rep):
    """Random keys of one DPC training run. rep 0 = the original run (controller init 20000 + s, starts 30000 + s);
    rep r > 0 = a fresh controller initialisation AND a fresh draw of the 1000 training starts."""
    return 20_000 + int(seed) + 1000 * int(rep), 30_000 + int(seed) + 1000 * int(rep)


def train_controllers(model, P, S0, seeds, plant, curriculum=CURRICULUM, n_x0=N_X0, log_every=250, dtype=jnp.float64,
                      verbose=True, reps=(0,)):
    """Train one controller (architecture ARCH) per (seed, repeat) through this model's dynamics, all together (vmapped).
    P, S0 are per seed; they are tiled over the repeats. Returns (C [len(reps) * S, ...], curve rows); the run order is
    repeat-major: index = k * S + i for repeat reps[k] and seed seeds[i]."""
    norm = model.get("norm")
    dyn = make_dynamics(model["kind"] if model.get("kind") == "true" else "surrogate", norm, plant)
    cast = lambda a: jnp.asarray(a, dtype) if jnp.issubdtype(jnp.asarray(a).dtype, jnp.floating) else a
    pairs = [(s, r) for r in reps for s in seeds]
    P = tmap(lambda a: jnp.concatenate([cast(a)] * len(reps), 0), P)
    S0 = jnp.asarray(np.concatenate([np.asarray(S0)] * len(reps)), dtype)
    X0 = jnp.stack([jax.random.uniform(jax.random.PRNGKey(run_keys(s, r)[1]), (n_x0, 1), dtype, -X0_RANGE, X0_RANGE)
                    for s, r in pairs])
    C = L.stack_trees([init_controller(run_keys(s, r)[0] - 20_000, dtype=dtype) for s, r in pairs])
    opt = optax.adamw(CTRL_LR)
    O = jax.vmap(opt.init)(C)
    curves, epoch, t0 = [], 0, time.time()
    for N, n_ep in curriculum:
        done = 0
        while done < n_ep:
            n = min(log_every, n_ep - done)
            step = get_dpc_trainer((model["name"], str(dtype), ARCH), dyn, N, n, opt, model.get("aggregate", "mean"), model.get("clip", 3.0))
            C, O, losses = step(C, O, P, S0, X0)
            losses = np.asarray(losses)
            done += n
            epoch += n
            for i, (s, r) in enumerate(pairs):
                curves.append(dict(model=model["name"], seed=s, rep=r, epoch=epoch, horizon=N, loss=float(losses[i, -1])))
        if verbose:
            last = np.array([r["loss"] for r in curves[-len(pairs):]])
            print(f"    horizon {N:3d}: {n_ep:5d} epochs | loss on its model: median {np.median(last):.4g}, "
                  f"max {np.max(last):.4g} | {time.time() - t0:.0f}s", flush=True)
    return C, curves


# ------------------------------------------------------------------------------------------------ #
# 4. Test on the TRUE plant                                                                        #
# ------------------------------------------------------------------------------------------------ #
def true_lqr_cost(xs, us, cfg=CONFIG):
    """The example's true_lqr_cost: (sum over steps of Q x^2 + R u^2, + Q_f x_N^2), mean over starts, / (N+1)."""
    stage = cfg["Q_x"] * xs[:-1] ** 2 + cfg["R_u"] * us ** 2
    return float((stage.sum(0) + cfg["Q_f"] * xs[-1] ** 2).mean() / xs.shape[0])


def per_start_cost(xs, us, cfg=CONFIG):
    """Unnormalised cost of each trajectory: sum_k (Q x_k^2 + R u_k^2) + Q_f x_N^2. xs [N+1, B], us [N, B]."""
    return (cfg["Q_x"] * xs[:-1] ** 2 + cfg["R_u"] * us ** 2).sum(0) + cfg["Q_f"] * xs[-1] ** 2


def lqr_saturated(plant, x0, N=CONFIG["N"], K=LQR_K):
    """Reference: analytic LQR u = clip(-K x, +-10) on the true plant."""
    xs, us = [np.asarray(x0, float)], []
    for _ in range(N):
        u = np.clip(-K * xs[-1], -U_BOUND, U_BOUND)
        us.append(u)
        xs.append(plant.A * xs[-1] + plant.B * u)
    return np.stack(xs), np.stack(us)


def _starts(n_runs, dtype):
    xe = jnp.asarray(np.repeat(X0_EVAL[None, :, None], n_runs, 0), dtype)
    xt = jnp.asarray(np.repeat(np.random.default_rng(12345).uniform(-X0_RANGE, X0_RANGE, (1, N_TEST, 1)), n_runs, 0), dtype)
    return xe, xt


def _roller(dyn, N=CONFIG["N"]):
    return jax.jit(jax.vmap(lambda c, p, s0, x0: rollout(c, dyn, p, s0, x0, N), in_axes=(0, 0, 0, 0)))


def evaluate_controllers(model, C, P, S0, seeds, plant, dtype=jnp.float64, kink=None, reps=(0,)):
    """Closed loops on the TRUE plant (and, for comparison, through the controller's own model) for the 22 evaluation
    starts and the 1000 typical starts, for every (seed, repeat) run in C (repeat-major order, as train_controllers).
    Returns (rows, {rep: trajectories})."""
    norm = model.get("norm")
    true_dyn = make_dynamics("true", plant=plant)
    own_dyn = true_dyn if model.get("kind") == "true" else make_dynamics("surrogate", norm, plant)
    cast = lambda a: jnp.asarray(a, dtype) if jnp.issubdtype(jnp.asarray(a).dtype, jnp.floating) else a
    pairs = [(s, r) for r in reps for s in seeds]
    P = tmap(lambda a: jnp.concatenate([cast(a)] * len(reps), 0), P)
    S0 = jnp.asarray(np.concatenate([np.asarray(S0)] * len(reps)), dtype)
    xe, xt = _starts(len(pairs), dtype)
    rt, ro = _roller(true_dyn), _roller(own_dyn)
    XE, UE = (np.asarray(a, float)[..., 0] for a in rt(C, P, S0, xe))          # [runs, N+1, 22], [runs, N, 22]
    XT, UT = (np.asarray(a, float)[..., 0] for a in rt(C, P, S0, xt))
    XO, UO = (np.asarray(a, float)[..., 0] for a in ro(C, P, S0, xt))          # same controller, its own model
    rows = []
    for i, (s, rep) in enumerate(pairs):
        J = per_start_cost(XE[i], UE[i])
        fin = np.abs(XE[i, -1])
        div = ~np.isfinite(XE[i]).all(0) | (np.nanmax(np.abs(XE[i]), 0) > DIVERGE)
        row = dict(model=model["name"], group=model["group"], seed=s, rep=rep,
                   cost_test_true=true_lqr_cost(XT[i], UT[i]), cost_test_own=true_lqr_cost(XO[i], UO[i]),
                   n_diverged=int(div.sum()), n_not_settled=int(((fin > SETTLED) & ~div).sum()),
                   final_abs_max=float(np.nanmax(fin)), final_abs_median=float(np.nanmedian(fin)),
                   test_not_settled=float(np.mean(np.abs(XT[i, -1]) > SETTLED)))
        for j, x0 in enumerate(X0_EVAL):
            row[f"J[{x0:+g}]"] = float(J[j])
        k = seeds.index(s)
        if kink is not None and np.isfinite(kink["rstar_x"][k]):
            zx, zu, rx, ru = (kink[q][k] for q in ("zmin_x", "zmin_u", "rstar_x", "rstar_u"))
            inside = ((XE[i, :-1] - zx) / rx) ** 2 + ((UE[i] - zu) / ru) ** 2 <= 1.0
            row["kink_share"] = float(inside.mean())
            row["kink_share_small"] = float(inside[:, np.abs(X0_EVAL) <= 0.2].mean())
        rows.append(row)
    trajs = {}
    for r in reps:
        idx = [i for i, (s, rr) in enumerate(pairs) if rr == r]
        trajs[r] = dict(XE=XE[idx].astype(np.float32), UE=UE[idx].astype(np.float32), seeds=np.array(seeds), x0=X0_EVAL)
    return rows, trajs


# ------------------------------------------------------------------------------------------------ #
# 5. The experiment, resumable per (model, repeat)                                                 #
# ------------------------------------------------------------------------------------------------ #
def append_rows(path, rows):
    """Append rows; if they bring columns the file lacks (e.g. kink_share after the oracle's rows), widen the file."""
    if rows and os.path.exists(path):
        import pandas as pd
        old = pd.read_csv(path)
        if set(rows[0]) - set(old.columns):
            pd.concat([old, pd.DataFrame(rows)], ignore_index=True).to_csv(path, index=False)
            return
    L.append_csv(path, rows)


def _read_runs(path):
    import pandas as pd
    df = pd.read_csv(path)
    if "rep" not in df:
        df["rep"] = 0                                   # files from before repeats existed = repeat 0
    df["rep"] = df["rep"].fillna(0).astype(int)
    return df


def done_runs(path, seeds):
    """{(model, rep)} with every seed finished."""
    if not os.path.exists(path):
        return set()
    df = _read_runs(path)
    return {(m, int(r)) for (m, r), g in df.groupby(["model", "rep"]) if set(seeds) <= set(g.seed)}


def rep_file(folder, name, rep):
    """rep 0 keeps the original file name, so a run made before repeats existed is reused as repeat 0."""
    return os.path.join(folder, safe(name) + ("" if rep == 0 else f"__rep{rep}") + ".npz")


def load_controllers(out_dir, name, rep):
    fn = rep_file(os.path.join(out_dir, "controllers"), name, rep)
    if not os.path.exists(fn):
        return None, None
    z = np.load(fn)
    return _unflat({k[2:]: z[k] for k in z.files if k.startswith("c:")}), list(z["seeds"])


def run_dpc(models, seeds, plant, sigma, kernel, out_dir, surr_cfgs, label_dir=None, curriculum=CURRICULUM,
            n_x0=N_X0, dtype=jnp.float64, M=2048, chunk=250, lam=1.0, reps=(0,), sur_dir=None):
    """sur_dir: folder whose surrogates/ (and labels) are reused; default out_dir. With ARCH != 'gru' pass the GRU study's
    folder as sur_dir and its arch_dir() as out_dir, so every architecture controls the SAME surrogates."""
    sur_dir = sur_dir or out_dir
    os.makedirs(os.path.join(out_dir, "controllers"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "trajectories"), exist_ok=True)
    res, cur = os.path.join(out_dir, "dpc_results.csv"), os.path.join(out_dir, "dpc_curves.csv")
    done = done_runs(res, seeds)
    todo = [(m["name"], r) for m in models for r in reps if (m["name"], r) not in done]
    print(f"{len(models)} models x {len(seeds)} seeds x repeats {list(reps)} -> {out_dir}  "
          f"({len(models) * len(reps) - len(todo)} model-repeats already done, skipped)")
    surr_rows = []
    for m in models:
        t0 = time.time()
        if m.get("kind") == "true":
            P, S0, met = L.stack_trees([L.init_params(s) for s in seeds]), np.ones(len(seeds)), None   # placeholder, unused
            m = dict(m, norm=None)
        else:
            cfg = surr_cfgs[m["surrogate"]]
            P, S0, met = train_surrogates(cfg, seeds, plant, sigma, kernel, sur_dir, label_dir, M, chunk, lam)
            m = dict(m, norm=cfg["norm"])
            for i, s in enumerate(seeds):
                surr_rows.append(dict(model=m["name"], seed=s, **{k: float(v[i]) for k, v in met.items()}))
        missing = [r for r in reps if (m["name"], r) not in done]
        if not missing:
            continue
        print(f"  DPC through '{m['name']}', repeats {missing}", flush=True)
        C, curves = train_controllers(m, P, S0, seeds, plant, curriculum, n_x0, dtype=dtype, reps=missing)
        rows, trajs = evaluate_controllers(m, C, P, S0, seeds, plant, dtype, met if m.get("norm") == "layer" else None, missing)
        for r in rows:
            r["train_s_per_seed"] = (time.time() - t0) / (len(seeds) * len(missing))
        append_rows(res, rows)
        append_rows(cur, curves)
        for k, rep in enumerate(missing):
            np.savez_compressed(rep_file(os.path.join(out_dir, "trajectories"), m["name"], rep), **trajs[rep])
            Ck = tmap(lambda a: a[k * len(seeds):(k + 1) * len(seeds)], C)
            np.savez_compressed(rep_file(os.path.join(out_dir, "controllers"), m["name"], rep), seeds=np.array(seeds),
                                **{"c:" + kk: v for kk, v in _flat(Ck).items()})
        med = lambda k: np.median([r[k] for r in rows])
        print(f"  -> {m['name']:24s} | true-plant test cost {med('cost_test_true'):.4g} (own model {med('cost_test_own'):.4g}) "
              f"| diverged {sum(r['n_diverged'] for r in rows)} | not settled {sum(r['n_not_settled'] for r in rows)} "
              f"| {time.time() - t0:.0f}s", flush=True)
    if surr_rows:
        import pandas as pd
        pd.DataFrame(surr_rows).to_csv(os.path.join(out_dir, "surrogates.csv"), index=False)


# ------------------------------------------------------------------------------------------------ #
# 6. Diagnostic: the reference controller run through each surrogate                              #
# ------------------------------------------------------------------------------------------------ #
def run_diagnostic(models, seeds, plant, out_dir, reps=(0,), dtype=jnp.float64, sur_dir=None):
    """For every surrogate model and repeat: take the controller trained through the TRUE plant (same seed and repeat)
    and run it through the surrogate. Two questions:
      1. Does the surrogate misbehave even under a good controller?  (oracle_on_model_* vs oracle_on_true)
      2. Which controller does the surrogate rate better: the good one, or the one DPC found through it?
         own_on_model < oracle_on_model -> the model prefers its own (possibly bad) controller: the MODEL misleads DPC.
         own_on_model > oracle_on_model -> a better controller existed for this model and DPC did not find it: an
         OPTIMISATION trap.
    Rows go to dpc_diag.csv (resumable per model and repeat)."""
    import pandas as pd
    path = os.path.join(out_dir, "dpc_diag.csv")
    done = set()
    if os.path.exists(path):
        d = pd.read_csv(path)
        done = {(m, int(r)) for (m, r), g in d.groupby(["model", "rep"]) if set(seeds) <= set(g.seed)}
    R = _read_runs(os.path.join(out_dir, "dpc_results.csv"))
    rows = []
    for m in models:
        if m.get("kind") == "true":
            continue
        cfg = surrogate_configs()[m["surrogate"]]
        P, S0, met, sseeds = load_surrogate(sur_dir or out_dir, m["surrogate"])
        idx = [sseeds.index(s) for s in seeds]
        P = tmap(lambda a: jnp.asarray(np.asarray(a)[idx], dtype), P)
        S0 = jnp.asarray(np.asarray(S0)[idx], dtype)
        roll = _roller(make_dynamics("surrogate", cfg["norm"], plant))
        for rep in reps:
            if (m["name"], rep) in done:
                continue
            Co, cs = load_controllers(out_dir, "true plant (oracle)", rep)
            if Co is None:
                continue
            Co = tmap(lambda a: jnp.asarray(np.asarray(a)[[cs.index(s) for s in seeds]], dtype), Co)
            xe, xt = _starts(len(seeds), dtype)
            XT, UT = (np.asarray(a, float)[..., 0] for a in roll(Co, P, S0, xt))
            XE, UE = (np.asarray(a, float)[..., 0] for a in roll(Co, P, S0, xe))
            for i, s in enumerate(seeds):
                g = R[(R.model == m["name"]) & (R.rep == rep) & (R.seed == s)]
                o = R[(R.model == "true plant (oracle)") & (R.rep == rep) & (R.seed == s)]
                fin = np.abs(XE[i, -1])
                div = ~np.isfinite(XE[i]).all(0) | (np.nanmax(np.abs(XE[i]), 0) > DIVERGE)
                rows.append(dict(model=m["name"], seed=s, rep=rep,
                                 oracle_on_model=true_lqr_cost(XT[i], UT[i]),
                                 oracle_on_true=float(o.cost_test_true.iloc[0]) if len(o) else np.nan,
                                 own_on_model=float(g.cost_test_own.iloc[0]) if len(g) else np.nan,
                                 own_on_true=float(g.cost_test_true.iloc[0]) if len(g) else np.nan,
                                 oracle_on_model_diverged=int(div.sum()),
                                 oracle_on_model_not_settled=int(((fin > SETTLED) & ~div).sum()),
                                 oracle_on_model_offset=float(np.nanmedian(np.abs(XE[i, -20:]).mean(0)))))
    if rows:
        append_rows(path, rows)
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()


def load_surrogate(out_dir, name):
    """(P, S0, metrics) of a saved surrogate configuration."""
    z = np.load(os.path.join(out_dir, "surrogates", safe(name) + ".npz"), allow_pickle=True)
    return (_unflat({k[2:]: z[k] for k in z.files if k.startswith("p:")}), z["S0"],
            {k[2:]: z[k] for k in z.files if k.startswith("m:")}, list(z["seeds"]))


def arch_dir(out_dir, arch=None):
    """The GRU study keeps its original folder; other architectures get a sibling folder (<out>_mlp, <out>_linear)
    so results never mix. Surrogates and labels are shared: pass sur_dir=out_dir to the training functions."""
    arch = arch or ARCH
    return out_dir if arch == "gru" else out_dir.rstrip("/") + "_" + arch


def linear_landscape(dyn, p, s0, x0, N, ks, bs, cfg=CONFIG):
    """DPC loss of the LINEAR controller u ~ k x + b (gain k in plant units, saturating at +-10) on a (k, b) grid, through `dyn` (a surrogate or the true
    plant), for one seed: the whole optimisation problem DPC faces, as a 2-D map. Returns loss [len(ks), len(bs)]."""
    ks, bs = jnp.asarray(ks, x0.dtype), jnp.asarray(bs, x0.dtype)

    def one(k, b):
        c = dict(k=jnp.full((1, 1), k / LIN_SCALE, x0.dtype), b=jnp.full((1,), b / LIN_SCALE, x0.dtype))
        return dpc_loss(c, dyn, p, s0, x0, N, cfg)
    f = jax.jit(jax.vmap(jax.vmap(one, in_axes=(None, 0)), in_axes=(0, None)))
    return np.asarray(f(ks, bs))


def gradient_audit(dyn_sur, dyn_true, p, s0, x0, N, ks, b=0.0, cfg=CONFIG):
    """Test of Corollary U5 on the linear controller u ~ k x + b: dL/dk through the SURROGATE and through the TRUE plant
    at the same gains k, same starts, same horizon. Where the two have opposite signs, DPC through the surrogate is pushed
    away from the direction that improves the true cost. Returns dict(k, g_sur, g_true, L_sur, L_true)."""
    ks = jnp.asarray(ks, x0.dtype)

    def loss(k, dyn):
        c = dict(k=jnp.full((1, 1), k / LIN_SCALE, x0.dtype), b=jnp.full((1,), b / LIN_SCALE, x0.dtype))
        return dpc_loss(c, dyn, p, s0, x0, N, cfg)
    vg = lambda dyn: jax.jit(jax.vmap(jax.value_and_grad(lambda k: loss(k, dyn))))
    Ls, gs = vg(dyn_sur)(ks)
    Lt, gt = vg(dyn_true)(ks)
    return dict(k=np.asarray(ks), g_sur=np.asarray(gs) / LIN_SCALE, g_true=np.asarray(gt) / LIN_SCALE,
                L_sur=np.asarray(Ls), L_true=np.asarray(Lt))


In [ ]:
%%writefile phase_core.py
"""phase_core.py - closed loops with a reference (setpoint), and what the controller sees along the way.

Three kinds of controller, all designed or trained from a surrogate F(x, u) and all run on the TRUE plant
x+ = 1.03 x + 0.01 u (and, for comparison, on the surrogate itself):

  lqr_fixed   LQR designed once at the setpoint: linearise the surrogate at its own equilibrium (r, u_hat(r)) where
              F(r, u_hat) = r, K = DARE(A_hat, B_hat), then u = clip(u_hat - K (x - r), +-10). Only the surrogate's
              slopes AT THE SETPOINT (and its equilibrium input u_hat) enter. Negative control for "the path matters".
  lqr_sched   Gain-scheduled LQR: at every step relinearise the surrogate at (x_k, u_{k-1}) and recompute K_k. The
              slopes ALONG THE PATH enter directly: a wrong-sign B_hat gives a wrong-sign gain.
  dpc         GRU controller trained through the surrogate (regulation controllers from dpc_core, or the
              reference-tracking controllers trained here: input (x - r, r), loss around the surrogate's equilibrium).

The reference controller is the same design on the TRUE plant (analytic LQR, or DPC trained through the true plant).
Everything uses the cost of dpc_core: sum_k Q (x_k - r)^2 + R (u_k - u_ss(r))^2 + Q_f (x_N - r)^2, u_ss(r) = 3 r.
"""
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
import optax
import level1_core as L
import dpc_core as DC

tmap = jax.tree_util.tree_map
Q, R, QF, N = DC.CONFIG["Q_x"], DC.CONFIG["R_u"], DC.CONFIG["Q_f"], DC.CONFIG["N"]
UB = DC.U_BOUND
R_EVAL = np.array([-2.0, -1.0, -0.5, -0.25, -0.1, 0.0, 0.1, 0.25, 0.5, 1.0, 2.0])   # setpoints evaluated
X0_TRACK = np.linspace(-3.0, 3.0, 9)                                              # starts per setpoint (metrics)
R_TRAIN = 2.0                                                                     # tracking DPC: r ~ U[-2, 2]
SETTLED = DC.SETTLED


# ------------------------------------------------------------------------------------------------ #
# 1. Models with one interface: F and its slopes on [S, n] arrays (S = seeds)                       #
# ------------------------------------------------------------------------------------------------ #
class Model:
    """A stacked surrogate (one per seed) or the true plant, evaluated on arrays X, U of shape [S, n]."""

    def __init__(self, P=None, S0=None, norm=None, seeds=None, plant=None, name=""):
        self.P, self.norm, self.seeds, self.plant, self.name = P, norm, list(seeds), plant, name
        self.true = P is None
        if not self.true:
            self.S0 = jnp.asarray(np.asarray(S0, float))
            self.ev = L.get_evaluator(norm)

    @classmethod
    def surrogate(cls, out_dir, name, seeds, display=""):
        P, S0, met, sseeds = DC.load_surrogate(out_dir, name)
        idx = [sseeds.index(s) for s in seeds]
        P = tmap(lambda a: jnp.asarray(np.asarray(a)[idx]), P)
        cfg = DC.surrogate_configs()[name]
        m = cls(P, np.asarray(S0)[idx], cfg["norm"], seeds, name=display or name)
        m.met = {k: np.asarray(v)[idx] for k, v in met.items()}
        return m

    @classmethod
    def true_plant(cls, plant, seeds):
        m = cls(seeds=seeds, plant=plant, name="true plant")
        m.met = {}
        return m

    def seed_view(self, i):
        """The same model restricted to seed index i (still stacked, S = 1)."""
        if self.true:
            return Model.true_plant(self.plant, [self.seeds[i]])
        m = Model(tmap(lambda a: a[i:i + 1], self.P), np.asarray(self.S0)[i:i + 1], self.norm, [self.seeds[i]], name=self.name)
        m.met = {k: v[i:i + 1] for k, v in self.met.items()}
        return m

    def fj(self, X, U):
        """F, dF/dx, dF/du on [S, n] arrays (numpy in, numpy out)."""
        X, U = np.asarray(X, float), np.asarray(U, float)
        if self.true:
            return self.plant.f(X, U), self.plant.Jx(X, U) + 0 * X, self.plant.Ju(X, U) + 0 * X
        out = [], [], []
        n = X.shape[1]
        step = max(1, 400_000 // max(1, X.shape[0]))
        for s in range(0, n, step):
            F, jx, ju = self.ev(self.P, jnp.asarray(X[:, s:s + step]), jnp.asarray(U[:, s:s + step]), self.S0)
            for o, v in zip(out, (F, jx, ju)):
                o.append(np.asarray(v))
        return tuple(np.concatenate(o, 1) for o in out)

    def kink(self):
        """(zmin_x, zmin_u, rstar_x, rstar_u) per seed, NaN for kink-free models."""
        if self.true or self.norm != "layer":
            return tuple(np.full(len(self.seeds), np.nan) for _ in range(4))
        return tuple(self.met[k] for k in ("zmin_x", "zmin_u", "rstar_x", "rstar_u"))


# ------------------------------------------------------------------------------------------------ #
# 2. The surrogate's own equilibrium: u_hat(r) with F(r, u_hat) = r                                 #
# ------------------------------------------------------------------------------------------------ #
def linear_fit(model, n=81):
    """Least-squares linear model F ~ a x + b u + c of each seed's surrogate over the data box (a data-only guess)."""
    xg, ug = L.grid(n)
    S = len(model.seeds)
    F, _, _ = model.fj(np.tile(xg, (S, 1)), np.tile(ug, (S, 1)))
    Phi = np.column_stack([xg, ug, np.ones_like(xg)])
    coef = np.linalg.lstsq(Phi, F.T, rcond=None)[0]          # [3, S]
    return coef[0], coef[1], coef[2]


def equilibrium(model, r, n_u=801):
    """u_hat [S, nr]: the surrogate's steady-state input for setpoints r (array [nr] or [S, nr]).
    F(r, .) - r can cross zero more than once near the kink (a wrong-sign gain makes F non-monotone in u): we take the
    root closest to the global linear fit's guess (r - a r - c) / b, which needs no true plant. Also returns the number
    of roots (> 1 = the surrogate has several equilibria at this setpoint) and the surrogate's slopes at (r, u_hat)."""
    S = len(model.seeds)
    r = np.broadcast_to(np.asarray(r, float), (S,) + np.shape(r)[-1:]).copy()
    nr = r.shape[1]
    if model.true:
        A, B = model.plant.A, model.plant.B
        uh = (r - A * r) / B
        return uh, np.ones_like(uh, int), np.full_like(uh, A), np.full_like(uh, B)
    a, b, c = linear_fit(model)
    guess = (r - a[:, None] * r - c[:, None]) / b[:, None]
    ug = np.linspace(-UB, UB, n_u)
    X = np.repeat(r, n_u, 1)
    U = np.tile(ug, (S, nr))
    G = model.fj(X, U)[0].reshape(S, nr, n_u) - r[..., None]
    uh = np.empty((S, nr))
    nroot = np.zeros((S, nr), int)
    for s in range(S):
        for j in range(nr):
            g = G[s, j]
            idx = np.where(np.sign(g[:-1]) * np.sign(g[1:]) <= 0)[0]
            if len(idx) == 0:
                uh[s, j] = ug[np.argmin(np.abs(g))]
                continue
            roots = ug[idx] - g[idx] * (ug[idx + 1] - ug[idx]) / np.where(g[idx + 1] != g[idx], g[idx + 1] - g[idx], 1.0)
            nroot[s, j] = len(roots)
            uh[s, j] = roots[np.argmin(np.abs(roots - guess[s, j]))]
    # refine with a few Newton steps on the chosen branch (stays on it: the bracket is one grid cell wide)
    for _ in range(3):
        F, _, ju = model.fj(r, uh)
        step = (F - r) / np.where(np.abs(ju) > 1e-9, ju, 1e-9)
        uh = np.clip(uh - np.clip(step, -0.03, 0.03), -UB, UB)
    _, ah, bh = model.fj(r, uh)
    return uh, nroot, ah, bh


# ------------------------------------------------------------------------------------------------ #
# 3. LQR closed loops (fixed at the setpoint, or rescheduled every step)                            #
# ------------------------------------------------------------------------------------------------ #
def run_lqr(design, plant_model, r, x0, kind, N=N):
    """design = the model the controller is designed from; plant_model = the model it runs on (true plant normally).
    r, x0: [S, n]. kind in {'fixed', 'sched'}. Returns xs [N+1, S, n], us [N, S, n], K [N, S, n]."""
    uh, nroot, ah, bh = equilibrium(design, r)
    Kf = L.dare_K(ah, bh)
    x, u_prev = np.asarray(x0, float).copy(), uh.copy()
    xs, us, Ks = [x], [], []
    for _ in range(N):
        if kind == "fixed":
            K = Kf
        else:
            _, a, b = design.fj(x, u_prev)
            K = L.dare_K(a, b)
        u = np.clip(uh - K * (x - r), -UB, UB)
        x = plant_model.fj(np.clip(x, -1e3, 1e3), u)[0] if not plant_model.true else plant_model.plant.f(x, u)
        xs.append(x), us.append(u), Ks.append(K)
        u_prev = u
    return np.stack(xs), np.stack(us), np.stack(Ks), dict(uh=uh, nroot=nroot, ah=ah, bh=bh, K_fixed=Kf)


# ------------------------------------------------------------------------------------------------ #
# 4. DPC closed loops (regulation controllers from dpc_core, or tracking controllers from below)    #
# ------------------------------------------------------------------------------------------------ #
def _gru_batch(C, h, inp):
    return jax.vmap(DC.ctrl_step)(C, h, inp)


def run_dpc(C, plant_model, x0, r=None, uh=None, N=N, track=False):
    """C stacked over seeds [S]. x0 (and r) [S, n]. Regulation controllers take x; tracking ones take (x - r, r).
    Returns xs [N+1, S, n], us [N, S, n]."""
    S, n = x0.shape
    x = np.asarray(x0, float).copy()
    h = jnp.zeros((S, n, DC.HIDDEN if "Wz" in C else 0))
    step = jax.jit(_gru_batch)
    xs, us = [x], []
    for _ in range(N):
        inp = np.stack([x - r, r], -1) if track else x[..., None]
        h, u = step(C, h, jnp.asarray(inp))
        u = np.asarray(u)[..., 0]
        x = plant_model.plant.f(x, u) if plant_model.true else plant_model.fj(np.clip(x, -1e3, 1e3), u)[0]
        xs.append(x), us.append(u)
    return np.stack(xs), np.stack(us)


# ------------------------------------------------------------------------------------------------ #
# 5. Metrics and what the controller sees along the path                                            #
# ------------------------------------------------------------------------------------------------ #
def track_cost(xs, us, r, plant):
    """True cost of each trajectory, measured around the TRUE equilibrium (r, u_ss(r))."""
    uss = plant.u_ss(r)
    return (Q * (xs[:-1] - r) ** 2 + R * (us - uss) ** 2).sum(0) + QF * (xs[-1] - r) ** 2


def track_metrics(xs, us, r, plant, ref_cost=None):
    J = track_cost(xs, us, r, plant)
    fin = np.abs(xs[-20:].mean(0) - r)
    div = ~np.isfinite(xs).all(0) | (np.nanmax(np.abs(xs), 0) > DC.DIVERGE)
    out = dict(cost=J, offset=fin, diverged=div, not_settled=(fin > SETTLED) & ~div)
    if ref_cost is not None:
        with np.errstate(divide="ignore", invalid="ignore"):
            out["gap"] = np.where(ref_cost > 1e-9, J / np.maximum(ref_cost, 1e-300) - 1.0, np.nan)
    return out


def seen_gain(model, xs, us):
    """The surrogate's slopes at the visited points (x_k, u_k): Jx_hat - A and Ju_hat / B. xs [N+1, S, n], us [N, S, n]."""
    Np, S, n = us.shape
    X = np.moveaxis(xs[:-1], 0, 1).reshape(S, -1)
    U = np.moveaxis(us, 0, 1).reshape(S, -1)
    _, jx, ju = model.fj(np.clip(X, -50, 50), U)
    back = lambda a: np.moveaxis(a.reshape(S, Np, n), 1, 0)
    return back(jx), back(ju)


def kink_visits(model, xs, us):
    """Share of steps inside each seed's closed-form kink ellipse."""
    zx, zu, rx, ru = model.kink()
    if not np.isfinite(rx).any():
        return np.full(xs.shape[1:], np.nan)
    inside = ((xs[:-1] - zx[None, :, None]) / rx[None, :, None]) ** 2 + ((us - zu[None, :, None]) / ru[None, :, None]) ** 2 <= 1
    return inside.mean(0)


def lqr_study(models, plant, refs=R_EVAL, starts=X0_TRACK, N=N):
    """Fixed and scheduled LQR from every surrogate, every seed, every (setpoint, start), on the TRUE plant.
    Returns a long table (one row per model, seed, controller, setpoint, start) and the trajectories."""
    import pandas as pd
    truth = Model.true_plant(plant, models[0].seeds)
    S = len(truth.seeds)
    rr, xx = np.meshgrid(refs, starts, indexing="ij")
    r = np.tile(rr.ravel(), (S, 1))
    x0 = np.tile(xx.ravel(), (S, 1))
    xs_o, us_o, _, _ = run_lqr(truth, truth, r, x0, "fixed", N)
    J_o = track_cost(xs_o, us_o, r, plant)
    rows, trajs = [], {}
    for m in models:
        zx, zu, rx, ru = m.kink()
        for kind in ("fixed", "sched"):
            t0 = time.time()
            xs, us, Ks, info = run_lqr(m, truth, r, x0, kind, N)
            met = track_metrics(xs, us, r, plant, J_o)
            share = kink_visits(m, xs, us)
            jx, ju = seen_gain(m, xs, us)
            wrong = (ju <= 0).mean(0)
            trajs[(m.name, kind)] = dict(xs=xs.astype(np.float32), us=us.astype(np.float32), r=r, x0=x0, info=info)
            for i, s in enumerate(m.seeds):
                d_eq = np.hypot((rr.ravel() - zx[i]) / rx[i], (plant.u_ss(rr.ravel()) - zu[i]) / ru[i]) if np.isfinite(rx[i]) else np.nan + 0 * rr.ravel()
                for j in range(r.shape[1]):
                    rows.append(dict(model=m.name, seed=s, controller=f"LQR {kind}", r=r[i, j], x0=x0[i, j],
                                     cost=met["cost"][i, j], ref_cost=J_o[i, j], gap=met["gap"][i, j],
                                     offset=met["offset"][i, j], diverged=bool(met["diverged"][i, j]),
                                     not_settled=bool(met["not_settled"][i, j]), kink_share=share[i, j],
                                     wrong_sign_share=wrong[i, j], uh=info["uh"][i, j], u_ss=plant.u_ss(r[i, j]),
                                     n_eq=info["nroot"][i, j], Ah=info["ah"][i, j], Bh=info["bh"][i, j],
                                     K_fixed=info["K_fixed"][i, j], setpoint_kink_dist=d_eq[j]))
            print(f"  {m.name:22s} LQR {kind:5s}: median gap {100 * np.nanmedian(met['gap']):8.3g} %, diverged "
                  f"{int(met['diverged'].sum())}/{met['diverged'].size} | {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows), trajs, dict(xs=xs_o, us=us_o, r=r, x0=x0, cost=J_o)


# ------------------------------------------------------------------------------------------------ #
# 6. Reference-tracking DPC: GRU input (x - r, r); loss around the surrogate's own equilibrium      #
# ------------------------------------------------------------------------------------------------ #
def track_keys(seed, rep):
    """controller init 50000 + s, training pairs 60000 + s (+ 1000 x repeat): separate from the regulation runs."""
    return 50_000 + int(seed) + 1000 * int(rep), 60_000 + int(seed) + 1000 * int(rep)


def rollout_track(c, dyn, p, s0, x0, r, N, hidden=None):
    h0 = jnp.zeros((x0.shape[0], DC.HIDDEN if "Wz" in c else 0), x0.dtype)

    def step(carry, _):
        h, x = carry
        h, u = DC.ctrl_step(c, h, jnp.concatenate([x - r, r], -1))
        return (h, dyn(p, s0, x, u)), (x, u)

    (h, xN), (xs, us) = jax.lax.scan(jax.checkpoint(step), (h0, x0), None, length=N)
    return jnp.concatenate([xs, xN[None]], 0), us


def track_loss(c, dyn, p, s0, x0, r, uh, N):
    """(sum_k mean(Q (x_k - r)^2 + R (u_k - u_hat(r))^2) + mean(Q_f (x_N - r)^2)) / N. u_hat = the SURROGATE's
    equilibrium input (the true plant's 3 r for the oracle): DPC only knows its model."""
    xs, us = rollout_track(c, dyn, p, s0, x0, r, N)
    stage = jnp.sum(jnp.mean(Q * (xs[:-1, :, 0] - r[:, 0]) ** 2 + R * (us[:, :, 0] - uh[:, 0]) ** 2, axis=1))
    return (stage + jnp.mean(QF * (xs[-1, :, 0] - r[:, 0]) ** 2)) / N


_TRACK_STEPS = {}


def _track_trainer(key, dyn, N, n_epochs, opt):
    k = (key, N, n_epochs)
    if k in _TRACK_STEPS:
        return _TRACK_STEPS[k]

    def one(c, o, p, s0, x0, r, uh):
        def st(carry, _):
            c, o = carry
            loss, g = jax.value_and_grad(track_loss)(c, dyn, p, s0, x0, r, uh, N)
            upd, o = opt.update(g, o, c)
            return (optax.apply_updates(c, upd), o), loss
        (c, o), losses = jax.lax.scan(st, (c, o), None, length=n_epochs)
        return c, o, losses

    f = jax.jit(jax.vmap(one))
    _TRACK_STEPS[k] = f
    return f


def train_track(model_spec, model, seeds, plant, curriculum=DC.CURRICULUM, n_pairs=DC.N_X0, dtype=jnp.float64,
                reps=(0,), log_every=250, verbose=True):
    """One tracking controller per (seed, repeat), trained through `model` (a Model; the true plant for the oracle).
    Training pairs: x0 ~ U[-3, 3], r ~ U[-2, 2]; u_hat(r) from the model's own equilibrium."""
    pairs = [(s, rp) for rp in reps for s in seeds]
    X0 = np.stack([np.asarray(jax.random.uniform(jax.random.PRNGKey(track_keys(s, rp)[1]), (n_pairs, 1), jnp.float64,
                                                 -DC.X0_RANGE, DC.X0_RANGE)) for s, rp in pairs])
    Rr = np.stack([np.asarray(jax.random.uniform(jax.random.PRNGKey(track_keys(s, rp)[1] + 7), (n_pairs, 1), jnp.float64,
                                                 -R_TRAIN, R_TRAIN)) for s, rp in pairs])
    t0 = time.time()
    UH = []
    for k, (s, rp) in enumerate(pairs):
        i = seeds.index(s)
        uh, nroot, _, _ = equilibrium(model.seed_view(i), Rr[k, :, 0][None], n_u=401)
        UH.append(uh[0][:, None])
    UH = np.stack(UH)
    if verbose:
        print(f"    surrogate equilibria for {len(pairs)} x {n_pairs} training setpoints: {time.time() - t0:.0f}s", flush=True)
    if model.true:
        dyn = DC.make_dynamics("true", plant=plant)
        P = L.stack_trees([L.init_params(s) for s, _ in pairs])
        S0 = jnp.ones(len(pairs), dtype)
    else:
        dyn = DC.make_dynamics("surrogate", model.norm, plant)
        idx = [seeds.index(s) for s, _ in pairs]
        P = tmap(lambda a: jnp.asarray(np.asarray(a)[idx], dtype), model.P)
        S0 = jnp.asarray(np.asarray(model.S0)[idx], dtype)
    C = L.stack_trees([DC.init_controller(track_keys(s, rp)[0] - 20_000, d_x=2, dtype=dtype) for s, rp in pairs])
    opt = optax.adamw(DC.CTRL_LR)
    O = jax.vmap(opt.init)(C)
    X0, Rr, UH = (jnp.asarray(a, dtype) for a in (X0, Rr, UH))
    curves, epoch = [], 0
    for Nh, n_ep in curriculum:
        done = 0
        while done < n_ep:
            n = min(log_every, n_ep - done)
            C, O, losses = _track_trainer((model_spec["name"], str(dtype), DC.ARCH), dyn, Nh, n, opt)(C, O, P, S0, X0, Rr, UH)
            losses = np.asarray(losses)
            done += n
            epoch += n
            for i, (s, rp) in enumerate(pairs):
                curves.append(dict(model=model_spec["name"], seed=s, rep=rp, epoch=epoch, horizon=Nh, loss=float(losses[i, -1])))
        if verbose:
            last = np.array([c["loss"] for c in curves[-len(pairs):]])
            print(f"    horizon {Nh:3d}: {n_ep:5d} epochs | loss on its model: median {np.median(last):.4g}, max {np.max(last):.4g}"
                  f" | {time.time() - t0:.0f}s", flush=True)
    return C, curves, pairs


def run_track_study(models, seeds, plant, out_dir, reps=(0,), curriculum=DC.CURRICULUM, n_pairs=DC.N_X0,
                    dtype=jnp.float64, refs=R_EVAL, starts=X0_TRACK, sur_dir=None):
    """Train tracking DPC through every model (resumable per model and repeat), evaluate on the TRUE plant for every
    (setpoint, start), and save rows (track_results.csv), controllers and trajectories (track_*/) in out_dir. The surrogates
    are read from sur_dir (default out_dir: the DPC run's folder, which holds surrogates/)."""
    import pandas as pd
    for d in ("track_controllers", "track_trajectories"):
        os.makedirs(os.path.join(out_dir, d), exist_ok=True)
    res = os.path.join(out_dir, "track_results.csv")
    done = set()
    if os.path.exists(res):
        df = pd.read_csv(res)
        done = {(m, int(r)) for (m, r), g in df.groupby(["model", "rep"]) if set(seeds) <= set(g.seed)}
    truth = Model.true_plant(plant, seeds)
    S = len(seeds)
    rr, xx = np.meshgrid(refs, starts, indexing="ij")
    r_e, x_e = np.tile(rr.ravel(), (S, 1)), np.tile(xx.ravel(), (S, 1))
    for spec in models:
        todo = [rp for rp in reps if (spec["name"], rp) not in done]
        if not todo:
            continue
        t0 = time.time()
        model = truth if spec.get("kind") == "true" else Model.surrogate(sur_dir or out_dir, spec["surrogate"], seeds, spec["name"])
        print(f"  tracking DPC through '{spec['name']}', repeats {todo}", flush=True)
        C, curves, pairs = train_track(spec, model, seeds, plant, curriculum, n_pairs, dtype, todo)
        rows = []
        for k, rp in enumerate(todo):
            Ck = tmap(lambda a: a[k * S:(k + 1) * S], C)
            xs, us = run_dpc(Ck, truth, x_e, r_e, track=True)
            xo, uo = (run_dpc(Ck, model, x_e, r_e, track=True) if not model.true else (xs, us))
            met = track_metrics(xs, us, r_e, plant)
            share = kink_visits(model, xs, us) if not model.true else np.full(xs.shape[1:], np.nan)
            if model.true:
                wrong, bh = np.zeros(xs.shape[1:]), np.full(xs.shape[1:], plant.B)
            else:                                   # what the surrogate says along the true path and at the setpoint
                _, ju = seen_gain(model, xs, us)
                wrong = (ju <= 0).mean(0)
                bh = equilibrium(model, r_e)[3]
            Jown = track_cost(xo, uo, r_e, plant)
            for i, s in enumerate(seeds):
                for j in range(r_e.shape[1]):
                    rows.append(dict(model=spec["name"], seed=s, rep=rp, r=r_e[i, j], x0=x_e[i, j], cost=met["cost"][i, j],
                                     cost_own=Jown[i, j], offset=met["offset"][i, j], diverged=bool(met["diverged"][i, j]),
                                     not_settled=bool(met["not_settled"][i, j]), kink_share=share[i, j],
                                     wrong_sign_share=wrong[i, j], Bh=bh[i, j]))
            np.savez_compressed(DC.rep_file(os.path.join(out_dir, "track_trajectories"), spec["name"], rp),
                                xs=xs.astype(np.float32), us=us.astype(np.float32), xo=xo.astype(np.float32),
                                uo=uo.astype(np.float32), r=r_e, x0=x_e, seeds=np.array(seeds))
            np.savez_compressed(DC.rep_file(os.path.join(out_dir, "track_controllers"), spec["name"], rp),
                                seeds=np.array(seeds), **{"c:" + kk: v for kk, v in DC._flat(Ck).items()})
        DC.append_rows(res, rows)
        DC.append_rows(os.path.join(out_dir, "track_curves.csv"), curves)
        g = pd.DataFrame(rows)
        print(f"  -> {spec['name']:24s} | median true cost {g.cost.median():.4g} | diverged {int(g.diverged.sum())} | "
              f"not settled {int(g.not_settled.sum())} of {len(g)} | {time.time() - t0:.0f}s", flush=True)
    return pd.read_csv(res) if os.path.exists(res) else None


def add_track_gaps(T):
    """gap = cost / cost of the oracle (same seed, repeat, setpoint, start) - 1; controller = 'DPC' (for the joint figures)."""
    o = T[T.model == "true plant (oracle)"][["seed", "rep", "r", "x0", "cost"]].rename(columns={"cost": "ref_cost"})
    T = T.drop(columns=[c for c in ("ref_cost", "gap") if c in T]).merge(o, on=["seed", "rep", "r", "x0"], how="left")
    T["gap"] = T.cost / T.ref_cost - 1.0
    T["controller"] = "DPC"
    return T


def load_track_controllers(out_dir, name, rep=0):
    fn = DC.rep_file(os.path.join(out_dir, "track_controllers"), name, rep)
    if not os.path.exists(fn):
        return None, None
    z = np.load(fn)
    return DC._unflat({k[2:]: z[k] for k in z.files if k.startswith("c:")}), list(z["seeds"])


def track_diagnostic(out_dir, sur_dir, seeds, plant, refs=R_EVAL, starts=X0_TRACK, rep=0):
    """Trap test for tracking DPC (like dpc_core.run_diagnostic): the tracking controller trained through the TRUE plant is run
    through each surrogate. Per model and seed (medians over setpoint x start pairs with |x0 - r| >= 0.5):
    ref_model_vs_true = its cost on the surrogate / on the true plant (~1: the surrogate judges a good controller correctly);
    trap = cost of DPC's own controller on the surrogate / the reference controller's (> 1: a better controller existed for
    this surrogate and DPC missed it)."""
    import pandas as pd
    Co, cs = load_track_controllers(out_dir, "true plant (oracle)", rep)
    idx = [cs.index(s) for s in seeds]
    Co = tmap(lambda a: a[np.array(idx)], Co)
    rr, xx = np.meshgrid(refs, starts, indexing="ij")
    r_e, x_e = np.tile(rr.ravel(), (len(seeds), 1)), np.tile(xx.ravel(), (len(seeds), 1))
    T = add_track_gaps(pd.read_csv(os.path.join(out_dir, "track_results.csv")))
    T = T[T.rep == rep]
    rows = []
    for m in DC.MODELS:
        if m.get("kind") == "true":
            continue
        model = Model.surrogate(sur_dir, m["surrogate"], seeds, m["name"])
        xs, us = run_dpc(Co, model, x_e, r_e, track=True)
        J = track_cost(xs, us, r_e, plant)
        for i, s in enumerate(seeds):
            g = T[(T.model == m["name"]) & (T.seed == s)].set_index(["r", "x0"]).loc[list(zip(r_e[i], x_e[i]))]
            far = np.abs(x_e[i] - r_e[i]) >= 0.5
            rows.append(dict(model=m["name"], seed=s, rep=rep, oracle_on_model=np.median(J[i, far]),
                             oracle_on_true=np.median(g.ref_cost.values[far]), own_on_model=np.median(g.cost_own.values[far]),
                             own_on_true=np.median(g.cost.values[far])))
    D = pd.DataFrame(rows)
    D["trap"] = D.own_on_model / D.oracle_on_model
    D["ref_model_vs_true"] = D.oracle_on_model / D.oracle_on_true
    D["gap_true"] = D.own_on_true / D.oracle_on_true - 1
    D.to_csv(os.path.join(out_dir, "track_diag.csv"), index=False)
    return D


# ------------------------------------------------------------------------------------------------ #
# 7. Training-time view of the trap: what DPC's gradients saw, and the landscape between the two   #
#    controllers (added 2026-10-23). Regulation task, 1000 training starts of repeat `rep`.          #
# ------------------------------------------------------------------------------------------------ #
def _training_starts(seeds, rep, dtype=jnp.float64):
    """The 1000 initial states each DPC run was trained on (key 30000 + s + 1000 rep), [S, 1000, 1]."""
    return jnp.stack([jax.random.uniform(jax.random.PRNGKey(DC.run_keys(s, rep)[1]), (DC.N_X0, 1), dtype,
                                         -DC.X0_RANGE, DC.X0_RANGE) for s in seeds])


def _stack_models(out_dir, name, seeds, dtype=jnp.float64):
    P, S0, met, ss = DC.load_surrogate(out_dir, name)
    idx = [ss.index(s) for s in seeds]
    P = tmap(lambda a: jnp.asarray(np.asarray(a)[idx], dtype), P)
    return P, jnp.asarray(np.asarray(S0)[idx], dtype), {k: np.asarray(v)[idx] for k, v in met.items()}


def adjoint_gradients(xs, us, jx, ju, plant, cfg=DC.CONFIG):
    """Per-step gradient of the regulation cost with respect to each input u_k along a GIVEN path, as backprop
    through the model would compute it: lam_N = 2 Q_f x_N, lam_k = 2 Q x_k + J_x(k) lam_{k+1}, g_k = 2 R u_k + J_u(k) lam_{k+1}.
    xs [N+1, B], us [N, B], jx, ju [N, B] = the slopes the model reports at (x_k, u_k). Returns g_model, g_true [N, B]
    (the same recursion with the true A, B), and lam_model [N+1, B]."""
    Q, R, QF = cfg["Q_x"], cfg["R_u"], cfg["Q_f"]
    N = us.shape[0]
    def run(JX, JU):
        lam = np.zeros_like(xs); g = np.zeros_like(us)
        lam[N] = 2 * QF * xs[N]
        for k in range(N - 1, -1, -1):
            g[k] = 2 * R * us[k] + JU[k] * lam[k + 1]
            lam[k] = 2 * Q * xs[k] + JX[k] * lam[k + 1]
        return g, lam
    gm, lm = run(jx, ju)
    gt, _ = run(np.full_like(jx, plant.A), np.full_like(ju, plant.B))
    return gm, gt, lm


def reference_path_colours(out_dir, seeds, plant, models=None, rep=0, dtype=jnp.float64, N=N):
    """What the surrogate's slopes look like along the REFERENCE controller's path (the controller trained through the true
    plant, run through each surrogate on the 1000 training starts): the slopes DPC had to learn from to reach a good policy.
    Per (model, seed): share of steps with wrong-sign J_u, with |J_u/B - 1| > 1, inside the kink ellipse; max |J_x|; share
    of steps with |J_x| > A + 0.5; and the adjoint per-step input gradients g_k through the model vs through the true plant
    along the same path: share of steps whose sign disagrees, and the share of the model's gradient energy sum g_k^2 that
    comes from steps inside the kink. Also the same statistics along DPC's OWN controller's path (what it ended up seeing)."""
    import pandas as pd
    models = models or [m for m in DC.MODELS if m.get("kind") != "true"]
    Co, cs = DC.load_controllers(out_dir, "true plant (oracle)", rep)
    Co = tmap(lambda a: jnp.asarray(np.asarray(a)[[cs.index(s) for s in seeds]], dtype), Co)
    x0 = _training_starts(seeds, rep, dtype)
    rows = []
    for m in models:
        cfg = DC.surrogate_configs()[m["surrogate"]]
        P, S0, met = _stack_models(out_dir, m["surrogate"], seeds, dtype)
        model = Model(P, np.asarray(S0), cfg["norm"], seeds, name=m["name"]); model.met = met
        roll = DC._roller(DC.make_dynamics("surrogate", cfg["norm"], plant), N)
        Cm, cm = DC.load_controllers(out_dir, m["name"], rep)
        Cm = tmap(lambda a: jnp.asarray(np.asarray(a)[[cm.index(s) for s in seeds]], dtype), Cm)
        zx, zu, rx, ru = model.kink()
        for who, C in (("reference", Co), ("own", Cm)):
            X, U = (np.asarray(a, float)[..., 0] for a in roll(C, P, S0, x0))     # [S, N+1, B], [S, N, B]
            xs, us = np.moveaxis(X, 0, 1), np.moveaxis(U, 0, 1)                      # [N+1, S, B]
            jx, ju = seen_gain(model, xs, us)
            for i, s in enumerate(seeds):
                g_m, g_t, lam = adjoint_gradients(xs[:, i], us[:, i], jx[:, i], ju[:, i], plant)
                if np.isfinite(rx[i]):
                    inside = ((xs[:-1, i] - zx[i]) / rx[i]) ** 2 + ((us[:, i] - zu[i]) / ru[i]) ** 2 <= 1
                else:
                    inside = np.zeros_like(us[:, i], bool)
                gain = ju[:, i] / plant.B
                e = g_m ** 2
                # the final approach: steps with |x| <= 0.2 (where regulation must end)
                near = np.abs(xs[:-1, i]) <= 0.2
                rows.append(dict(model=m["name"], seed=s, rep=rep, path=who,
                                 peak_Ju=float(met["Ju_sup_rel"][i]), rstar_x=rx[i], rstar_u=ru[i],
                                 wrong_sign=float((gain <= 0).mean()), bad_gain=float((np.abs(gain - 1) > 1).mean()),
                                 wrong_sign_near=float((gain[near] <= 0).mean()) if near.any() else np.nan,
                                 kink_share=float(inside.mean()), Jx_max=float(np.abs(jx[:, i]).max()),
                                 Jx_big=float((np.abs(jx[:, i]) > plant.A + 0.5).mean()),
                                 grad_sign_flip=float((np.sign(g_m) != np.sign(g_t)).mean()),
                                 grad_sign_flip_near=float((np.sign(g_m) != np.sign(g_t))[near].mean()) if near.any() else np.nan,
                                 grad_energy_kink=float(e[inside].sum() / max(e.sum(), 1e-300)),
                                 grad_energy_wrong=float(e[gain <= 0].sum() / max(e.sum(), 1e-300)),
                                 grad_rms_ratio=float(np.sqrt(e.mean() / max((g_t ** 2).mean(), 1e-300))),
                                 cost_model=float(DC.true_lqr_cost(X[i], U[i])),
                                 diverged=float((np.abs(X[i]).max(0) > DC.DIVERGE).mean())))
    return pd.DataFrame(rows)


def _interp(Ca, Cb, a):
    return tmap(lambda u, v: (1 - a) * u + a * v, Ca, Cb)


def cost_interpolation(out_dir, seeds, plant, model_spec, rep=0, alphas=np.linspace(0, 1, 21), dtype=jnp.float64, N=N):
    """The regulation loss along the straight line in weight space from DPC's own controller (alpha = 0) to the reference
    controller (alpha = 1), evaluated through the surrogate (what DPC minimised) and through the true plant, on the 1000
    training starts. A barrier in the surrogate loss with none in the true loss is the optimisation trap in cost units.
    The surrogate loss is also split into the stage cost accrued inside / outside the kink ellipse. Also returns the norm
    of the surrogate-loss gradient at both ends (is the reference controller a stationary point of the surrogate loss?)."""
    import pandas as pd
    cfg = DC.surrogate_configs()[model_spec["surrogate"]]
    P, S0, met = _stack_models(out_dir, model_spec["surrogate"], seeds, dtype)
    Co, cs = DC.load_controllers(out_dir, "true plant (oracle)", rep)
    Co = tmap(lambda a: jnp.asarray(np.asarray(a)[[cs.index(s) for s in seeds]], dtype), Co)
    Cm, cm = DC.load_controllers(out_dir, model_spec["name"], rep)
    Cm = tmap(lambda a: jnp.asarray(np.asarray(a)[[cm.index(s) for s in seeds]], dtype), Cm)
    x0 = _training_starts(seeds, rep, dtype)
    dyn_s, dyn_t = DC.make_dynamics("surrogate", cfg["norm"], plant), DC.make_dynamics("true", plant=plant)
    roll_s, roll_t = DC._roller(dyn_s, N), DC._roller(dyn_t, N)
    gS = jax.jit(jax.vmap(jax.grad(lambda c, p, s0, x: DC.dpc_loss(c, dyn_s, p, s0, x, N))))
    gnorm = lambda C: np.asarray(jax.vmap(lambda g: jnp.sqrt(sum(jnp.sum(l ** 2) for l in jax.tree_util.tree_leaves(g))))(gS(C, P, S0, x0)))
    zx, zu, rx, ru = (met[k] for k in ("zmin_x", "zmin_u", "rstar_x", "rstar_u"))
    Q, R, QF = DC.CONFIG["Q_x"], DC.CONFIG["R_u"], DC.CONFIG["Q_f"]
    rows = []
    for a in alphas:
        C = _interp(Cm, Co, float(a))
        Xs, Us = (np.asarray(v, float)[..., 0] for v in roll_s(C, P, S0, x0))
        Xt, Ut = (np.asarray(v, float)[..., 0] for v in roll_t(C, P, S0, x0))
        for i, s in enumerate(seeds):
            stage = Q * Xs[i, :-1] ** 2 + R * Us[i] ** 2                       # [N, B]
            if np.isfinite(rx[i]):
                inside = ((Xs[i, :-1] - zx[i]) / rx[i]) ** 2 + ((Us[i] - zu[i]) / ru[i]) ** 2 <= 1
            else:
                inside = np.zeros_like(stage, bool)
            rows.append(dict(model=model_spec["name"], seed=s, rep=rep, alpha=float(a),
                             loss_model=DC.true_lqr_cost(Xs[i], Us[i]), loss_true=DC.true_lqr_cost(Xt[i], Ut[i]),
                             loss_model_inside=float((stage * inside).sum(0).mean() / Xs.shape[1]),
                             loss_model_outside=float((stage * ~inside).sum(0).mean() / Xs.shape[1]),
                             kink_share_model=float(inside.mean()),
                             offset_model=float(np.median(np.abs(Xs[i, -20:]).mean(0))),
                             diverged_model=float((np.abs(Xs[i]).max(0) > DC.DIVERGE).mean())))
    D = pd.DataFrame(rows)
    g0, g1 = gnorm(Cm), gnorm(Co)
    G = pd.DataFrame(dict(model=model_spec["name"], seed=seeds, rep=rep, gradnorm_own=g0, gradnorm_reference=g1))
    return D, G


def closed_loop_stability(out_dir, seeds, plant, models=None, rep=0, dtype=jnp.float64, N=N, n_starts=64):
    """Is the GOOD controller closed-loop stable according to the surrogate? For the reference controller (trained through
    the true plant) run through each surrogate: the surrogate's slopes at its resting point (median over the last 20 steps:
    J_u_hat / B and J_x_hat), and the Lyapunov exponent of the closed loop (controller + model) along each rollout,
    (1/N) log || d(h_N, x_N) / d x_0 ||, averaged over n_starts training starts. Through the true plant the oracle gives
    log(0.927) = -0.076 per step. A positive exponent through the surrogate means the model's slopes make the good
    controller locally unstable at the setpoint, so backpropagating the loss through N steps multiplies the gradient by
    exp(N lyap): the loss VALUE can stay right while its GRADIENT explodes. Also reported for DPC's own controller."""
    import pandas as pd
    models = models or [m for m in DC.MODELS if m.get("kind") != "true"]
    Co, cs = DC.load_controllers(out_dir, "true plant (oracle)", rep)
    Co = tmap(lambda a: jnp.asarray(np.asarray(a)[[cs.index(s) for s in seeds]], dtype), Co)
    x0 = _training_starts(seeds, rep, dtype)[:, :n_starts]

    def make_lyap(dyn):
        def final(c, p, s0, x0_scalar):
            xs, us = DC.rollout(c, dyn, p, s0, x0_scalar[None, None], N)
            return xs[-1, 0, 0]
        def one(c, p, s0, x0s):            # x0s [n]
            # sensitivity of the final state to the initial state (true plant: 0.927^N); log / N
            d = jax.vmap(jax.grad(final, argnums=3), in_axes=(None, None, None, 0))(c, p, s0, x0s)
            return jnp.log(jnp.abs(d) + 1e-300) / N
        return jax.jit(jax.vmap(one))

    rows = []
    for m in models:
        cfg = DC.surrogate_configs()[m["surrogate"]]
        P, S0, met = _stack_models(out_dir, m["surrogate"], seeds, dtype)
        model = Model(P, np.asarray(S0), cfg["norm"], seeds, name=m["name"]); model.met = met
        dyn = DC.make_dynamics("surrogate", cfg["norm"], plant)
        roll = DC._roller(dyn, N)
        lyap = make_lyap(dyn)
        Cm, cm = DC.load_controllers(out_dir, m["name"], rep)
        Cm = tmap(lambda a: jnp.asarray(np.asarray(a)[[cm.index(s) for s in seeds]], dtype), Cm)
        for who, C in (("reference", Co), ("own", Cm)):
            X, U = (np.asarray(a, float)[..., 0] for a in roll(C, P, S0, x0))
            ly = np.asarray(lyap(C, P, S0, x0[..., 0]))                                   # [S, n]
            xs, us = np.moveaxis(X, 0, 1), np.moveaxis(U, 0, 1)
            jx, ju = seen_gain(model, xs, us)
            for i, s in enumerate(seeds):
                rows.append(dict(model=m["name"], seed=s, rep=rep, path=who, peak_Ju=float(met["Ju_sup_rel"][i]),
                                 rest_x=float(np.median(np.abs(X[i, -20:]))), rest_gain=float(np.median(ju[-20:, i] / plant.B)),
                                 rest_Jx=float(np.median(jx[-20:, i])), rest_gain_min=float(np.min(ju[-20:, i] / plant.B)),
                                 lyap_mean=float(np.mean(ly[i])), lyap_max=float(np.max(ly[i])),
                                 lyap_pos_share=float(np.mean(ly[i] > 0))))
    return pd.DataFrame(rows)


In [ ]:
%%writefile phase_report.py
"""phase_report.py - time-ordered (x_k, u_k) phase plots over the surrogate's slope maps, and setpoint studies."""
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, SymLogNorm
import level1_core as L
import phase_core as PC

TRAJ_COL = ["#1baf7a", "#eda100", "#6250d6", "#0b0b0b"]       # starts (validated categorical slots + ink)
MODEL_COL = {"LN, MSE (3000)": "#6f6e69", "LN, MSE (12000)": "#eda100", "LN, J_x + J_u (3000)": "#1baf7a",
             "LN, two-stage": "#6250d6", "no norm, MSE": "#eb6834", "static norm, MSE": "#2a78d6",
             "true plant (oracle)": "#0b0b0b"}
MODEL_ORDER = ["LN, MSE (3000)", "LN, MSE (12000)", "LN, J_x + J_u (3000)", "LN, two-stage", "no norm, MSE", "static norm, MSE"]
TIME_MARKS = (1, 2, 5, 10, 20, 50)
LIN_X, LIN_U = 0.25, 1.0          # symlog axes: linear within |x| <= 0.25, |u| <= 1


def _save(fig, out_dir, name):
    d = os.path.join(out_dir, "figs")
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, name + ".png"), bbox_inches="tight", dpi=150)
    fig.savefig(os.path.join(d, name + ".pdf"), bbox_inches="tight")


def _symgrid(lo, hi, t, n):
    """n points between lo and hi, uniform on a symlog axis with linear range +-t."""
    f = lambda v: np.sign(v) * np.log1p(np.abs(v) / t)
    g = lambda w: np.sign(w) * t * np.expm1(np.abs(w))
    return g(np.linspace(f(lo), f(hi), n))


def slope_maps(model1, xlim, ulim, n=301, sym=True):
    """Grids and fields for one seed (model1 = Model.seed_view): J_u / B, J_x - A, F(x, u) - x."""
    xs = _symgrid(*xlim, LIN_X, n) if sym else np.linspace(*xlim, n)
    us = _symgrid(*ulim, LIN_U, n) if sym else np.linspace(*ulim, n)
    X, U = np.meshgrid(xs, us)
    F, jx, ju = model1.fj(X.ravel()[None], U.ravel()[None])
    sh = X.shape
    return dict(X=X, U=U, Ju=(ju[0] / 0.01).reshape(sh), Jx=(jx[0] - 1.03).reshape(sh), G=(F[0] - X.ravel()).reshape(sh))


def phase_panel(ax, maps, trajs, plant, r=None, uh=None, field="Ju", kink=None, sym=True, own=None, legend=False,
                label_time=False):
    """Background: the surrogate's input gain / true (field 'Ju') or state-gain error (field 'Jx'). Overlays: true
    equilibrium line, the surrogate's equilibrium curve F(x, u) = x, the setpoint, the kink ellipse, and the time-ordered
    trajectories [(xs, us), ...] (solid = true plant; `own` = the same controller run through the surrogate, dashed)."""
    X, U = maps["X"], maps["U"]
    if field == "Ju":
        im = ax.pcolormesh(X, U, np.clip(maps["Ju"], -3, 5), cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-3, vcenter=1, vmax=5),
                           shading="auto", rasterized=True)
        if np.any(maps["Ju"] < 0):
            ax.contour(X, U, maps["Ju"], levels=[0], colors="#0b0b0b", linewidths=0.6)
    else:
        im = ax.pcolormesh(X, U, np.clip(maps["Jx"], -1, 1), cmap="PuOr_r", norm=SymLogNorm(1e-2, vmin=-1, vmax=1, base=10),
                           shading="auto", rasterized=True)
    ax.contour(X, U, maps["G"], levels=[0], colors="#0b0b0b", linewidths=1.0, linestyles="--")
    xl = np.array([X.min(), X.max()])
    ax.plot(xl, plant.u_ss(xl), color="#52514e", lw=1.0, ls=":")
    if kink is not None and np.isfinite(kink[2]):
        t = np.linspace(0, 2 * np.pi, 200)
        ax.plot(kink[0] + kink[2] * np.cos(t), kink[1] + kink[3] * np.sin(t), color="#0b0b0b", lw=0.8, ls="-.")
    for k, (xs, us) in enumerate(trajs):
        col = TRAJ_COL[k % len(TRAJ_COL)]
        xk, uk = xs[:-1], us
        ok = np.isfinite(xk) & (np.abs(xk) < 50)
        xk, uk = xk[ok], uk[ok]
        ax.plot(xk, uk, color="white", lw=3.2, zorder=4, solid_capstyle="round")
        ax.plot(xk, uk, color=col, lw=1.6, zorder=5)
        if own is not None:
            xo, uo = own[k][0][:-1], own[k][1]
            ok2 = np.isfinite(xo) & (np.abs(xo) < 50)
            ax.plot(xo[ok2], uo[ok2], color=col, lw=1.0, ls=(0, (3, 2)), zorder=5)
        ax.plot(xk[0], uk[0], "o", ms=7, mfc="white", mec=col, mew=2, zorder=6)
        for j, km in enumerate(TIME_MARKS):
            if km < len(xk):
                ax.plot(xk[km], uk[km], "o", ms=2.5 + 0.7 * j, color=col, mec="white", mew=0.6, zorder=6)
                if label_time and k == 0 and km in (1, 5, 20):
                    ax.annotate(f"k={km}", (xk[km], uk[km]), xytext=(4, 4), textcoords="offset points", fontsize=6.5,
                                color="#0b0b0b", zorder=7, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.7))
        ax.plot(xk[-1], uk[-1], "X", ms=7, color=col, mec="white", mew=0.8, zorder=6)
    if r is not None:
        ax.plot(r, plant.u_ss(r), "*", ms=14, mfc="white", mec="#0b0b0b", mew=1.2, zorder=7)
        if uh is not None:
            ax.plot(r, uh, "D", ms=5, mfc="#0b0b0b", mec="white", mew=0.6, zorder=7)
    if sym:
        ax.set_xscale("symlog", linthresh=LIN_X)
        ax.set_yscale("symlog", linthresh=LIN_U)
    ax.set_xlim(X.min(), X.max())
    ax.set_ylim(U.min(), U.max())
    ax.grid(False)
    if legend:
        ax.legend(handles=legend_handles(own is not None), fontsize=6.5, loc="upper right", framealpha=0.9, frameon=True)
    return im


def legend_handles(own=False, starts=None):
    from matplotlib.lines import Line2D
    if True:
        h = [Line2D([], [], color="#52514e", ls=":", label="true equilibria u = u_ss(x)"),
             Line2D([], [], color="#0b0b0b", ls="--", label="surrogate's equilibria F(x, u) = x"),
             Line2D([], [], color="#0b0b0b", ls="-.", lw=0.8, label="closed-form kink ellipse"),
             Line2D([], [], marker="*", ls="", ms=11, mfc="white", mec="#0b0b0b", label="setpoint (r, u_ss(r))"),
             Line2D([], [], marker="D", ls="", ms=5, mfc="#0b0b0b", mec="white", label="surrogate's u_hat(r)"),
             Line2D([], [], marker="o", ls="", ms=7, mfc="white", mec="#52514e", mew=2, label="start (k = 0)"),
             Line2D([], [], marker="X", ls="", ms=7, color="#52514e", label="k = 199"),
             Line2D([], [], color="#52514e", lw=1.6, marker="o", ms=3, label="true plant; dots at k = 1, 2, 5, 10, 20, 50")]
        if own:
            h.append(Line2D([], [], color="#52514e", lw=1.0, ls=(0, (3, 2)), label="same controller through the surrogate"))
        if starts is not None:
            h += [Line2D([], [], color=TRAJ_COL[k % len(TRAJ_COL)], lw=2.2, label=f"start x0 = {x:g}") for k, x in enumerate(starts)]
        return h


def fig_legend(fig, own=False, starts=None, bbox=(0.905, 0.55)):
    fig.legend(handles=legend_handles(own, starts), fontsize=7, loc="upper left", bbox_to_anchor=bbox, frameon=False)


def strip_panels(ax_x, ax_g, trajs, seen, r, plant, cols=TRAJ_COL):
    """Below a phase panel: tracking error x_k - r, and the surrogate input gain the path visits (J_u_hat / B)."""
    for k, (xs, us) in enumerate(trajs):
        kk = np.arange(len(us)) + 1
        ax_x.plot(np.arange(len(xs)) + 1, xs - r, color=cols[k % len(cols)], lw=1.4)
        ax_g.plot(kk, seen[k], color=cols[k % len(cols)], lw=1.4)
    for a in (ax_x, ax_g):
        a.set_xscale("log")
        a.axhline(0, color="#8a8984", lw=0.8)
    ax_x.set_yscale("symlog", linthresh=0.01)
    lo, hi = ax_x.get_ylim()
    ax_x.set_ylim(max(lo, -10), min(hi, 10))
    ax_x.set_ylabel("x_k - r\n(lin. within +-0.01)", fontsize=7)
    ax_g.set_yscale("symlog", linthresh=1)
    ax_g.axhline(1, color="#8a8984", lw=0.8, ls=":")
    ax_g.axhspan(-1e3, 0, color="#f3d9d9", zorder=0, lw=0)
    ax_g.set_ylim(min(-3, ax_g.get_ylim()[0]), max(5, ax_g.get_ylim()[1]))
    ax_g.set_ylabel("J_u_hat / B seen\n(red band = wrong sign)", fontsize=7)
    ax_g.set_xlabel("step k + 1", fontsize=7)
    for a in (ax_x, ax_g):
        a.tick_params(labelsize=6.5)


def _window(trajs, r, plant, pad=1.25):
    xs = np.concatenate([t[0][np.isfinite(t[0]) & (np.abs(t[0]) < 50)] for t in trajs] + [np.array([r, -0.3, 0.3])])
    us = np.concatenate([t[1][np.isfinite(t[1])] for t in trajs] + [np.array([plant.u_ss(r), -1.2, 1.2])])
    xl = (max(-4.0, xs.min() * pad - 0.05), min(4.0, xs.max() * pad + 0.05))
    ul = (max(-10.5, us.min() * pad - 0.2), min(10.5, us.max() * pad + 0.2))
    return xl, ul


def _gap_text(J, J0, x0, r, xs):
    """Per-start cost gap (only for starts at least 0.5 from the setpoint: nearer ones have almost no transient, so any
    steady offset looks like a huge relative gap) and the steady offset |x - r| averaged over the last 20 steps."""
    J, J0, x0 = np.ravel(J), np.ravel(J0), np.ravel(x0)
    parts = []
    for j in range(len(J)):
        if abs(x0[j] - r) < 0.5 or J0[j] < 1e-9:
            parts.append("-")
        elif not np.isfinite(J[j]) or J[j] / J0[j] > 1e3:
            parts.append("diverged" if not np.isfinite(J[j]) or J[j] / J0[j] > 1e6 else f"{J[j] / J0[j] - 1:.0f}x")
        else:
            parts.append(f"{100 * (J[j] / J0[j] - 1):.2g}%")
    fin = np.abs(np.asarray(xs)[-20:].reshape(20, -1).mean(0) - r)
    fin = fin[np.isfinite(fin) & (fin < 10)]
    off = f"{np.median(fin):.2g}" if len(fin) else "n/a"
    return "cost gap per start: " + ", ".join(parts) + f"\nsteady offset |x - r| (median over starts) {off}"


def fig_phase_lqr(model, seed, plant, out_dir, refs=(0.0, 0.25, 1.0), starts=(-2.5, -0.5, 0.5, 2.5), kinds=("fixed", "sched"),
                  name=None):
    """One surrogate, one seed. Columns = setpoints; for each LQR design: phase panel + two strips."""
    i = model.seeds.index(seed)
    m1 = model.seed_view(i)
    truth = PC.Model.true_plant(plant, [seed])
    kink = tuple(v[0] for v in m1.kink())
    nk = len(kinds)
    fig = plt.figure(figsize=(5.0 * len(refs), 6.3 * nk))
    gs = fig.add_gridspec(4 * nk, len(refs), height_ratios=[3.2, 1, 1, 0.45] * nk, hspace=0.5, wspace=0.28, top=0.91, right=0.86)
    im = None
    for a, kind in enumerate(kinds):
        for j, r in enumerate(refs):
            rr = np.full((1, len(starts)), r)
            x0 = np.array(starts, float)[None]
            xs, us, Ks, info = PC.run_lqr(m1, truth, rr, x0, kind)
            trajs = [(xs[:, 0, k], us[:, 0, k]) for k in range(len(starts))]
            _, ju = PC.seen_gain(m1, xs, us)
            seen = [ju[:, 0, k] / plant.B for k in range(len(starts))]
            xl, ul = _window(trajs, r, plant)
            maps = slope_maps(m1, xl, ul)
            ax = fig.add_subplot(gs[4 * a, j])
            im = phase_panel(ax, maps, trajs, plant, r, info["uh"][0, 0], kink=kink, label_time=False)
            J0 = PC.track_cost(*PC.run_lqr(truth, truth, rr, x0, "fixed")[:2], rr, plant)
            J = PC.track_cost(xs, us, rr, plant)
            gap = np.where(J0 > 1e-9, J / np.maximum(J0, 1e-12) - 1, np.nan)[0]
            ttl = (f"LQR {'designed at the setpoint' if kind == 'fixed' else 'rescheduled every step'}, r = {r:g}\n"
                   f"B_hat/B at setpoint {info['bh'][0, 0] / plant.B:.2g}, K = {info['K_fixed'][0, 0]:.3g} (true 10.3); "
                   f"{info['nroot'][0, 0]} surrogate equilibri{'um' if info['nroot'][0, 0] == 1 else 'a'}\n" + _gap_text(J, J0, x0, r, xs))
            ax.set_title(ttl, fontsize=7.5)
            ax.set_xlabel("x (symlog)", fontsize=7.5)
            ax.set_ylabel("u (symlog)", fontsize=7.5)
            ax.tick_params(labelsize=6.5)
            strip_panels(fig.add_subplot(gs[4 * a + 1, j]), fig.add_subplot(gs[4 * a + 2, j]), trajs, seen, r, plant)
    cax = fig.add_axes([0.88, 0.76, 0.012, 0.17])
    fig.colorbar(im, cax=cax).set_label("surrogate / true input gain\n(1 = correct, < 0 = wrong sign)", fontsize=7.5)
    fig_legend(fig, starts=starts, bbox=(0.875, 0.72))
    fig.suptitle(f"{model.name}, seed {seed}: LQR from the surrogate, run on the true plant. Starts x0 = "
                 + ", ".join(f"{s:g}" for s in starts) + " (colours). Axes are symmetric-log so the kink and the far setpoints "
                 "fit in one view.", fontsize=9, y=0.985)
    _save(fig, out_dir, name or f"figP1_lqr_phase_{PC.DC.safe(model.name)}_seed{seed}")
    return fig


def fig_phase_atlas(models, seed, plant, out_dir, controller="fixed", refs=(-1.0, 0.0, 0.25, 1.0),
                    starts=(-2.5, -0.5, 0.5, 2.5), name=None):
    """Rows = surrogates, columns = setpoints, one seed and one LQR design: where the closed loop runs over each slope map."""
    fig, axes = plt.subplots(len(models), len(refs), figsize=(4.2 * len(refs), 3.7 * len(models)), squeeze=False)
    im = None
    for a, model in enumerate(models):
        i = model.seeds.index(seed)
        m1 = model.seed_view(i)
        truth = PC.Model.true_plant(plant, [seed])
        kink = tuple(v[0] for v in m1.kink())
        for j, r in enumerate(refs):
            rr = np.full((1, len(starts)), r)
            x0 = np.array(starts, float)[None]
            xs, us, Ks, info = PC.run_lqr(m1, truth, rr, x0, controller)
            trajs = [(xs[:, 0, k], us[:, 0, k]) for k in range(len(starts))]
            maps = slope_maps(m1, (-3.2, 3.2), (-10.5, 10.5), n=241)
            ax = axes[a, j]
            im = phase_panel(ax, maps, trajs, plant, r, info["uh"][0, 0], kink=kink, label_time=False)
            J0 = PC.track_cost(*PC.run_lqr(truth, truth, rr, x0, "fixed")[:2], rr, plant)
            J = PC.track_cost(xs, us, rr, plant)
            ok = J0[0] > 1e-9
            ax.set_title(f"{model.name}, r = {r:g}\nB_hat/B at setpoint {info['bh'][0, 0] / plant.B:.2g}\n"
                         + _gap_text(J, J0, x0, r, xs).replace("cost gap per start", "gap").replace("steady offset |x - r| (median over starts)", "offset"), fontsize=7)
            ax.tick_params(labelsize=6.5)
            if j == 0:
                ax.set_ylabel("u (symlog)", fontsize=7.5)
            if a == len(models) - 1:
                ax.set_xlabel("x (symlog)", fontsize=7.5)
    fig.subplots_adjust(top=1 - 1.15 / fig.get_figheight(), hspace=0.5, wspace=0.22)
    fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01).set_label("surrogate / true input gain", fontsize=8)
    what = "designed once at the setpoint" if controller == "fixed" else "rescheduled every step"
    fig.suptitle(f"Seed {seed}: LQR {what}, on the true plant, for four setpoints (star). Dashed = surrogate's equilibria, "
                 "dotted = true equilibria; starts " + ", ".join(f"{s:g}" for s in starts), fontsize=9, y=1 - 0.2 / fig.get_figheight())
    _save(fig, out_dir, name or f"figP2_atlas_lqr_{controller}_seed{seed}")
    return fig


def _gap_table(T, min_dist=0.5):
    """Gaps only for starts at least min_dist from the setpoint: with x0 ~ r the reference cost is ~ 0 and a steady offset
    of 0.01 alone looks like a huge relative gap."""
    T = T[(np.abs(T.x0 - T.r) >= min_dist)].copy()
    T["failed"] = (T.gap > 0.10) | T.diverged
    return T


def fig_gap_vs_setpoint(T, out_dir, controllers=("LQR fixed", "LQR sched"), name="figP3_gap_vs_setpoint", kink_halfwidth=None,
                        title_extra=""):
    """Cost gap and steady offset against the setpoint, per surrogate (median over seeds and starts, IQR band over seeds)."""
    T = _gap_table(T)
    fig, axes = plt.subplots(3, len(controllers), figsize=(6.2 * len(controllers), 9.2), squeeze=False, sharex=True)
    for j, c in enumerate(controllers):
        g = T[T.controller == c]
        for mname in [m for m in MODEL_ORDER if m in set(g.model)]:
            h = g[g.model == mname]
            per_seed = h.groupby(["r", "seed"]).agg(gap=("gap", "median"), off=("offset", "median"), fail=("failed", "mean"))
            for row, (col, f, lab) in enumerate((("gap", 100, "cost gap, median over starts (%)"),
                                                ("fail", 100, "starts failed: gap > 10 % or diverged (%)"),
                                                ("off", 1, "steady offset |x - r| (median)"))):
                st = per_seed[col].groupby("r")
                x = st.median().index.values
                ax = axes[row, j]
                ax.plot(x, f * st.median().values, "o-", ms=3.5, color=MODEL_COL[mname], lw=1.8, label=mname)
                ax.fill_between(x, f * st.quantile(0.25).values, f * st.quantile(0.75).values, color=MODEL_COL[mname],
                                alpha=0.12, lw=0)
                ax.set_ylabel(lab, fontsize=8)
        axes[0, j].set_title(c.replace("LQR fixed", "LQR designed at the setpoint").replace("LQR sched", "LQR rescheduled every step")
                             .replace("DPC", "tracking DPC"), fontsize=9)
        axes[0, j].set_yscale("log")
        axes[2, j].set_yscale("log")
        axes[2, j].set_xlabel("setpoint r (symlog, linear within +-0.1)")
        for a in axes[:, j]:
            a.set_xscale("symlog", linthresh=0.1)
            if kink_halfwidth is not None:
                for v in (-kink_halfwidth, kink_halfwidth):
                    a.axvline(v, color="#0b0b0b", lw=0.8, ls="-.")
    axes[0, 0].legend(fontsize=7)
    axes[2, 0].axhline(PC.SETTLED, color="#d03b3b", lw=0.8, ls=":")
    fig.suptitle("Where the setpoint sits relative to the kink: true-plant cost gap and steady offset against the setpoint r "
                 "(line = median over seeds, band = inter-quartile range over seeds; dash-dot lines = +- median LN-MSE kink half-width; "
                 "starts within 0.5 of r excluded from the gap)" + title_extra, fontsize=8.5, y=1.0)
    fig.tight_layout()
    _save(fig, out_dir, name)
    return fig


def fig_setpoint_distance(T, out_dir, controller="LQR fixed", name="figP4_setpoint_distance"):
    """Per (seed, setpoint) of the LayerNorm surrogates: distance of the setpoint from the kink centre (in kink half-widths,
    measured in the (x, u) plane at the setpoint's true equilibrium input) against (left) the error of the surrogate's input
    gain there, which is what an LQR designed at the setpoint inherits, and (right) the resulting true-plant cost gap."""
    T = _gap_table(T)
    g = T[(T.controller == controller) & T.model.str.startswith("LN")]
    P = g.groupby(["model", "seed", "r"]).agg(d=("setpoint_kink_dist", "first"), bh=("Bh", "first"), gap=("gap", "median"),
                                              div=("diverged", "mean")).reset_index()
    P["e"] = np.abs(P.bh / 0.01 - 1)
    far = P[P.d > 1]
    slope, icpt = np.polyfit(np.log(far.d), np.log(far.e + 1e-6), 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    for mname in [m for m in MODEL_ORDER if m in set(P.model)]:
        h = P[P.model == mname]
        ok, bad = h[h.bh > 0], h[h.bh <= 0]
        ax[0].scatter(ok.d, ok.e, s=16, color=MODEL_COL[mname], label=mname, edgecolor="white", linewidth=0.4)
        ax[0].scatter(bad.d, bad.e, s=34, marker="x", color=MODEL_COL[mname], linewidth=1.4)
        hv = h[h["div"] > 0]
        hn = h[h["div"] == 0]
        ax[1].scatter(hn.d, 100 * np.maximum(hn.gap, 1e-4), s=16, color=MODEL_COL[mname], edgecolor="white", linewidth=0.4)
        ax[1].scatter(hv.d, np.full(len(hv), 3e4), s=34, marker="x", color=MODEL_COL[mname], linewidth=1.4)
    dd = np.geomspace(0.05, 150, 50)
    ax[0].plot(dd, np.exp(icpt) * dd ** slope, "k--", lw=1, label=f"fit beyond 1 half-width: slope {slope:.2f}")
    ax[0].axhline(1, color="#d03b3b", lw=0.8, ls=":")
    ax[0].text(160, 1.25, "above: can have the wrong sign", fontsize=7, color="#d03b3b", ha="right")
    ax[0].set_xscale("log"); ax[0].set_yscale("log")
    ax[0].set_xlabel("setpoint distance from the kink centre (kink half-widths)")
    ax[0].set_ylabel("|surrogate input gain at the setpoint / B - 1|")
    ax[0].set_title("The gain an LQR designed at the setpoint inherits: error falls ~ 1 / distance\n"
                    "(x = wrong sign; one dot per LayerNorm surrogate, seed and setpoint)", fontsize=9)
    ax[1].set_xscale("log"); ax[1].set_yscale("log")
    ax[1].set_xlabel("setpoint distance from the kink centre (kink half-widths)")
    ax[1].set_ylabel("cost gap, median over starts (%)")
    ax[1].set_title(f"{controller.replace('LQR fixed', 'LQR designed at the setpoint')}: true-plant cost gap\n"
                    "(x at the top = diverged on at least one start)", fontsize=9)
    ax[1].axhline(10, color="#d03b3b", lw=0.8, ls=":")
    for a in ax:
        a.axvline(1, color="#8a8984", lw=0.8, ls="--")
    ax[0].legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    _save(fig, out_dir, name)
    return fig, slope


def fig_phase_dpc(models, seed, plant, out_dir, C_by_model, starts=(-2.5, -0.5, 0.5, 2.5), r=0.0, track=False, name=None,
                  fields=("Ju", "Jx")):
    """DPC controllers (regulation, or tracking at setpoint r): rows = surrogates, per field a phase panel with the true-plant
    path (solid) and the same controller run through its surrogate (dashed), plus the two strips."""
    fig = plt.figure(figsize=(4.6 * len(models), 3.9 * len(fields) + 3.0))
    gs = fig.add_gridspec(len(fields) + 2, len(models), height_ratios=[3.2] * len(fields) + [1, 1], hspace=0.4, wspace=0.3,
                          top=0.92, right=0.85)
    ims = {}
    for j, model in enumerate(models):
        i = model.seeds.index(seed)
        m1 = model.seed_view(i)
        truth = PC.Model.true_plant(plant, [seed])
        C, cseeds = C_by_model[model.name]
        C1 = PC.tmap(lambda a: a[cseeds.index(seed):cseeds.index(seed) + 1], C)
        x0 = np.array(starts, float)[None]
        rr = np.full_like(x0, r)
        xs, us = PC.run_dpc(C1, truth, x0, rr, track=track)
        xo, uo = PC.run_dpc(C1, m1, x0, rr, track=track)
        trajs = [(xs[:, 0, k], us[:, 0, k]) for k in range(len(starts))]
        own = [(xo[:, 0, k], uo[:, 0, k]) for k in range(len(starts))]
        _, ju = PC.seen_gain(m1, xs, us)
        seen = [ju[:, 0, k] / plant.B for k in range(len(starts))]
        xl, ul = _window(trajs + own, r, plant)
        maps = slope_maps(m1, xl, ul)
        kink = tuple(v[0] for v in m1.kink())
        uh = PC.equilibrium(m1, np.array([r]))[0][0, 0]
        J0 = PC.track_cost(*PC.run_lqr(truth, truth, rr, x0, "fixed")[:2], rr, plant)
        J = PC.track_cost(xs, us, rr, plant)
        for f_i, field in enumerate(fields):
            ax = fig.add_subplot(gs[f_i, j])
            ims[field] = phase_panel(ax, maps, trajs, plant, r, uh, field=field, kink=kink, own=own,
                                     label_time=False)
            ttl = f"{model.name}, seed {seed}" + ("\n" + _gap_text(J, J0, x0, r, xs).replace("cost gap", "cost gap vs true-plant LQR") if f_i == 0 else "")
            ax.set_title(ttl, fontsize=7.5)
            ax.tick_params(labelsize=6.5)
            ax.set_xlabel("x (symlog)", fontsize=7)
            if j == 0:
                ax.set_ylabel(("input gain map" if field == "Ju" else "state gain map") + "\nu (symlog)", fontsize=7.5)
        strip_panels(fig.add_subplot(gs[len(fields), j]), fig.add_subplot(gs[len(fields) + 1, j]), trajs, seen, r, plant)
    if "Ju" in ims:
        cb = fig.colorbar(ims["Ju"], cax=fig.add_axes([0.87, 0.80, 0.01, 0.12]))
        cb.set_label("J_u_hat / B\n(< 0 = wrong sign)", fontsize=7)
    if "Jx" in ims:
        cb = fig.colorbar(ims["Jx"], cax=fig.add_axes([0.87, 0.62, 0.01, 0.12]))
        cb.set_label("J_x_hat - A\n(state gain error)", fontsize=7)
    fig_legend(fig, own=True, starts=starts, bbox=(0.86, 0.58))
    what = f"tracking DPC, setpoint r = {r:g}" if track else "regulation DPC (setpoint 0)"
    fig.suptitle(f"{what}: time-ordered (x_k, u_k) on the true plant (solid) and through the controller's own surrogate (dashed). "
                 "Top: surrogate / true input gain; middle: state-gain error (purple = A_hat too small, orange = too large); "
                 "strips: tracking error and the input gain the true path visits", fontsize=8.5, y=0.985)
    _save(fig, out_dir, name or f"figP5_dpc_phase_seed{seed}{'_r' + str(r) if track else ''}")
    return fig


CTRL_STYLE = {"LQR fixed": ("#0b0b0b", "-", "o", "LQR designed at the setpoint"),
              "LQR sched": ("#d03b3b", "--", "s", "LQR rescheduled every step"),
              "DPC": ("#2a78d6", "-.", "^", "tracking DPC")}
WS_BINS = [-0.01, 0.0, 0.02, 0.1, 1.0]
WS_LABELS = ["0", "0-2 %", "2-10 %", "> 10 %"]


def path_table(T, good=0.2):
    """Per run (LayerNorm surrogates only, starts >= 0.5 from the setpoint): share of the true path's steps where the surrogate
    says the input gain has the wrong sign, and whether the gain at the setpoint is accurate (|B_hat/B - 1| < good)."""
    T = _gap_table(T)
    T = T[T.model.str.startswith("LN")].copy()
    T["ws_bin"] = pd.cut(T.wrong_sign_share, WS_BINS, labels=WS_LABELS)
    T["setpoint_ok"] = np.abs(T.Bh / 0.01 - 1) < good
    return T


def fig_path_vs_setpoint(T, out_dir, name="figP6_path_vs_setpoint", good=0.2, catastrophic=False, drop_broken=False):
    """Does it matter where the closed loop goes, or only where it ends? Failure rate against the share of the path spent where
    the surrogate's input gain has the wrong sign, for each controller; right: only runs whose setpoint gain is accurate.
    catastrophic=True counts only gap > 100 % or divergence (not the 10-100 % gaps that steady offsets cause);
    drop_broken=True leaves out (controller, model, seed) combinations that fail catastrophically on more than half their runs
    (a controller broken everywhere, e.g. a DPC policy trapped in training, says nothing about the path)."""
    P = path_table(T, good)
    P["cat"] = (P.gap > 1.0) | P.diverged
    if drop_broken:
        P = P[P.groupby(["controller", "model", "seed"]).cat.transform("mean") <= 0.5]
    if catastrophic:
        P["failed"] = P["cat"]
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
    rows = []
    for a, (sub, ttl) in enumerate(((P, "all runs"), (P[P.setpoint_ok], f"only runs whose setpoint gain is within {100 * good:.0f} % of B"))):
        for c in [c for c in CTRL_STYLE if c in set(sub.controller)]:
            col, ls, mk, lab = CTRL_STYLE[c]
            g = sub[sub.controller == c].groupby("ws_bin", observed=False).agg(fail=("failed", "mean"), n=("failed", "size"))
            ax[a].plot(range(len(g)), 100 * g.fail.values, color=col, ls=ls, marker=mk, lw=1.8, ms=6, label=lab)
            for b, row in g.iterrows():
                rows.append(dict(subset=ttl, controller=c, wrong_sign_share=b, failed_pct=100 * row.fail, n=int(row.n)))
        ax[a].set_xticks(range(len(WS_LABELS)))
        ax[a].set_xticklabels(WS_LABELS)
        ax[a].set_xlabel("share of the true path's steps where the surrogate says J_u has the wrong sign")
        ax[a].set_title(ttl + " (run counts per bin: see the table)", fontsize=9)
    ax[0].set_ylabel("runs failed: " + ("gap > 100 % or diverged" if catastrophic else "gap > 10 % or diverged") + " (%)")
    ax[0].legend(fontsize=7.5)
    fig.suptitle("Does the path's colour matter, or only the setpoint's? LayerNorm surrogates, setpoints and starts (>= 0.5 from "
                 "the setpoint)" + ("; catastrophic failures only" if catastrophic else "")
                 + ("; broken controllers (seeds failing everywhere) left out" if drop_broken else ""), fontsize=9.5)
    fig.tight_layout()
    _save(fig, out_dir, name)
    return fig, pd.DataFrame(rows)


def fig_seed_setpoint_grid(T, out_dir, controllers=("LQR fixed", "LQR sched", "DPC"),
                           models=("LN, MSE (3000)", "LN, J_x + J_u (3000)", "LN, two-stage", "static norm, MSE"),
                           name="figP8_seed_setpoint_grid"):
    """Rows = seeds, columns = setpoints; colour = share of starts that fail (gap > 10 % or diverged). Separates failures
    that follow the SETPOINT (a vertical band at the kink) from failures that follow the SEED (whole rows: the controller
    itself is broken, whatever the setpoint)."""
    from matplotlib.colors import ListedColormap
    G = _gap_table(T)
    controllers = [c for c in controllers if c in set(G.controller)]
    models = [m for m in models if m in set(G.model)]
    fig, axes = plt.subplots(len(controllers), len(models), figsize=(3.3 * len(models) + 1.2, 3.0 * len(controllers)),
                             squeeze=False, constrained_layout=True)
    cmap = matplotlib.colormaps["Reds"]
    im = None
    for i, c in enumerate(controllers):
        for j, m in enumerate(models):
            ax = axes[i, j]
            g = G[(G.controller == c) & (G.model == m)]
            M = 100 * g.groupby(["seed", "r"]).failed.mean().unstack()
            im = ax.imshow(M.values, cmap=cmap, vmin=0, vmax=100, aspect="auto")
            ax.set_xticks(range(M.shape[1]))
            ax.set_xticklabels([f"{v:g}" for v in M.columns], fontsize=6, rotation=90)
            ax.set_yticks(range(M.shape[0]))
            ax.set_yticklabels(M.index, fontsize=6.5)
            ax.grid(False)
            n_bad_rows = int((M.mean(1) > 50).sum())
            ax.set_title(f"{c.replace('LQR fixed', 'LQR at setpoint').replace('LQR sched', 'LQR rescheduled').replace('DPC', 'tracking DPC')}"
                         f"\n{m}: {M.values.mean():.0f} % failed; seeds failing > half: {n_bad_rows}", fontsize=7.5)
            if j == 0:
                ax.set_ylabel("seed", fontsize=7.5)
            if i == len(controllers) - 1:
                ax.set_xlabel("setpoint r", fontsize=7.5)
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01).set_label("starts failed (%)", fontsize=8)
    fig.suptitle("Is a failure tied to the setpoint (vertical band near r = 0) or to the seed (whole rows)? "
                 "Share of starts (>= 0.5 from r) with gap > 10 % or divergence", fontsize=9)
    _save(fig, out_dir, name)
    return fig


In [ ]:
import os, sys, time, shutil
try:                                   # Colab: the DPC run's folder on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DPC_OUT_ROOT"] = "/content/drive/MyDrive/dpc_out"
except ImportError:
    pass
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import jax.numpy as jnp
sys.path.insert(0, os.getcwd())
import importlib, level1_core as L, dpc_core as DC, phase_core as PC, phase_report as PR
for m in (L, DC, PC, PR):
    importlib.reload(m)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
OUT = DC.out_root()
need = [os.path.join(OUT, "surrogates", DC.safe(m["surrogate"]) + ".npz") for m in DC.MODELS if m.get("surrogate")]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise FileNotFoundError("This notebook needs the DPC run's folder (surrogates/ and controllers/). Missing:\n" + "\n".join(missing)
                            + f"\nPoint DPC_OUT_ROOT at it, or unzip dpc_out_backup.zip into {OUT}.")
print("surrogates and controllers from", OUT)

## Settings

In [ ]:
QUICK  = False                 # True: tiny tracking-DPC training (~5 min) into a separate folder, to check the setup
ARCH   = "gru"                 # tracking controller: "gru" (original), "mlp" (memoryless u = pi(x - r, r)), "linear"
SEEDS  = list(range(10))
DTYPE  = "float64"             # "float64" on an A100 (or CPU); "float32" on L4 / G4 / T4
PHASE_SEEDS = [9, 2, 6, 0]     # seeds drawn in the phase figures: 9, 2 = narrow kink (LN MSE fails), 6 = widest kink, 0
PHASE_REFS  = (0.0, 0.25, 1.0) # setpoints in the per-seed figures: inside, at the edge of, and far from the kink
STARTS      = (-2.5, -0.5, 0.5, 2.5)
RUN_TRACK   = True             # train the tracking DPC controllers
TRACK_REPS  = [0]              # tracking-DPC repeats (add 1, 2 later for robustness, like the regulation run)

PLANT = L.Plant(A=1.03, B=0.01, eps_nl=0.0)
SUR_DIR = OUT                  # surrogates and the GRU regulation controllers are read from here
DC.ARCH = ARCH
OUT = DC.arch_dir(OUT, ARCH)   # "gru" writes into the DPC folder; other architectures into <folder>_<arch>
CURRICULUM, N_PAIRS = DC.CURRICULUM, DC.N_X0
if QUICK:
    CURRICULUM, N_PAIRS, TRACK_REPS = [(5, 20), (10, 20), (20, 20), (50, 20), (100, 10), (200, 10)], 128, [0]
    OUT = OUT.rstrip("/") + "_phase_quick"
if DTYPE == "float32":
    jax.config.update("jax_default_matmul_precision", "highest")
    OUT = OUT.rstrip("/") + "_f32"
os.makedirs(OUT, exist_ok=True)
FDTYPE = jnp.float64 if DTYPE == "float64" else jnp.float32
SURR = {m["name"]: PC.Model.surrogate(SUR_DIR, m["surrogate"], SEEDS, m["name"]) for m in DC.MODELS if m.get("kind") != "true"}
print("phase results go to", OUT)

## 1 - LQR from every surrogate, every seed, setpoint and start (no training; ~1-2 min)
`gap` = true-plant cost / cost of the analytic LQR of the true plant from the same start - 1. Starts within 0.5 of the
setpoint are left out of the gap (they have almost no transient, so any steady offset looks like a huge relative gap).

In [ ]:
fn = os.path.join(OUT, "phase_lqr.csv")
if os.path.exists(fn):
    TL = pd.read_csv(fn)
else:
    TL, _, _ = PC.lqr_study(list(SURR.values()), PLANT)
    TL.to_csv(fn, index=False)
G = PR._gap_table(TL)
for c in ("LQR fixed", "LQR sched"):
    g = G[G.controller == c]
    print(f"{c}: runs failed (gap > 10 % or diverged), % of the runs at each setpoint (10 seeds x the 8 starts at least 0.5 from r)")
    display((100 * g.groupby(["model", "r"]).failed.mean().unstack()).round(0).loc[[m for m in PR.MODEL_ORDER if m in set(g.model)]])

## Figure P3 - Where the setpoint sits relative to the kink
Cost gap, failure rate and steady offset against the setpoint (median over seeds, band = IQR over seeds). Dash-dot lines =
+- the median kink half-width of LayerNorm MSE. Expectation: the LayerNorm surrogates fail near r = 0 and recover once the
setpoint is a few kink widths away; the kink-free models are flat.

In [ ]:
HW = float(np.median(SURR["LN, MSE (3000)"].met["rstar_x"]))
PR.fig_gap_vs_setpoint(TL, OUT, kink_halfwidth=HW); plt.show()

## Figure P4 - The gain an LQR designed at the setpoint inherits, against the setpoint's distance from the kink
One dot per LayerNorm surrogate, seed and setpoint. Left: |B_hat / B - 1| at the setpoint (x = wrong sign), with a power-law
fit beyond one half-width: the petal theory predicts slope -1. Right: the resulting cost gap (x at the top = diverged).

In [ ]:
fig, slope = PR.fig_setpoint_distance(TL, OUT); plt.show()
print(f"fitted log-log slope beyond one kink half-width: {slope:.2f} (petal theory: -1)")

## Figure P6 - Does the path's colour matter, or only the setpoint's?
Failure rate against the share of the true path's steps where the surrogate says the input gain has the wrong sign. Left: all
runs; right: only runs whose setpoint gain is accurate (within 20 % of B), which removes the setpoint effect. Expectation: the
fixed LQR is flat on the right (it never uses the path's slopes); the rescheduled LQR rises. The tracking-DPC line is added
after section 4.

In [ ]:
fig, tab = PR.fig_path_vs_setpoint(TL, OUT); plt.show()
display(tab.round(1))

## Figure P1 - Per seed: LQR phase portraits at three setpoints
Columns = setpoints (inside, at the edge of, far from the kink); top block = LQR designed at the setpoint, bottom block =
rescheduled every step. Title: the surrogate's gain and LQR gain K at the setpoint (true K = 10.3), the number of surrogate
equilibria there (3 = the dashed curve folds: the kink makes F non-monotone in u), per-start cost gap and steady offset.

In [ ]:
for s in PHASE_SEEDS:
    for mname in ("LN, MSE (3000)", "LN, two-stage"):
        PR.fig_phase_lqr(SURR[mname], s, PLANT, OUT, refs=PHASE_REFS, starts=STARTS); plt.show()

## Figure P2 - Atlas: all six surrogates x four setpoints, one seed, one LQR design

In [ ]:
ms = [SURR[k] for k in PR.MODEL_ORDER]
for kind in ("fixed", "sched"):
    PR.fig_phase_atlas(ms, PHASE_SEEDS[0], PLANT, OUT, controller=kind, refs=(-1.0, 0.0, 0.25, 1.0), starts=STARTS); plt.show()

## Figure P5 - Regulation DPC (the controllers of the DPC notebook, repeat 0), setpoint 0
Solid = the controller on the true plant; dashed = the same controller run through its own surrogate (what DPC was optimising).
Top row: input-gain map; middle row: state-gain error map (purple = A_hat too small, orange = too large).

In [ ]:
REG = ["LN, MSE (3000)", "LN, J_x + J_u (3000)", "LN, two-stage", "static norm, MSE"]
C_REG = {k: DC.load_controllers(SUR_DIR, k, 0) for k in REG}
for s in PHASE_SEEDS:
    PR.fig_phase_dpc([SURR[k] for k in REG], s, PLANT, OUT, C_REG, starts=STARTS); plt.show()

## 4 - Tracking DPC: train through every model, test on the true plant at every setpoint
One line per horizon of the curriculum (the loss on the controller's own model), then a summary line per model.
Resumable: a model and repeat already in `track_results.csv` is skipped.

In [ ]:
if RUN_TRACK:
    t0 = time.time()
    TT = PC.run_track_study(DC.MODELS, SEEDS, PLANT, OUT, reps=TRACK_REPS, curriculum=CURRICULUM, n_pairs=N_PAIRS,
                            dtype=FDTYPE, sur_dir=SUR_DIR)
    print(f"done in {(time.time() - t0) / 60:.1f} min")
TT = pd.read_csv(os.path.join(OUT, "track_results.csv"))
TT = PC.add_track_gaps(TT)
GT = PR._gap_table(TT[TT.model != "true plant (oracle)"])
print("tracking DPC: runs failed (gap > 10 % or diverged), % of seeds x starts (x repeats) at each setpoint")
display((100 * GT.groupby(["model", "r"]).failed.mean().unstack()).round(0).loc[[m for m in PR.MODEL_ORDER if m in set(GT.model)]])
print("median steady offset |x - r| by setpoint")
display(GT.groupby(["model", "r"]).offset.median().unstack().round(3).loc[[m for m in PR.MODEL_ORDER if m in set(GT.model)]])

## Figure P3 with tracking DPC, and Figure P6 with tracking DPC
The tracking-DPC gap is measured against the tracking controller trained through the true plant (same seed, repeat,
setpoint and start); the LQR gaps against the true plant's LQR.

In [ ]:
ALL = pd.concat([TL, TT[TT.model != "true plant (oracle)"]], ignore_index=True)
PR.fig_gap_vs_setpoint(ALL, OUT, controllers=("LQR fixed", "LQR sched", "DPC"), kink_halfwidth=HW,
                       name="figP3b_gap_vs_setpoint_with_dpc"); plt.show()
fig, tab = PR.fig_path_vs_setpoint(ALL, OUT, name="figP6b_path_vs_setpoint_with_dpc"); plt.show()
display(tab.round(1))

## Figure P8 - Is a failure tied to the setpoint or to the seed?
Rows = seeds, columns = setpoints, colour = share of starts that fail. LQR failures form a vertical band at the kink (they
follow the setpoint); a tracking-DPC failure, if any, fills whole rows (the trained policy itself is broken, whatever the
setpoint).

In [ ]:
PR.fig_seed_setpoint_grid(ALL, OUT); plt.show()

## Figure P6c - Path colours, catastrophic failures only
Counts only failures with gap > 100 % or divergence (the 10-100 % gaps are mostly steady offsets: a value error of the
surrogate amplified by 1/B, not the kink) and leaves out controllers broken on more than half of their runs.

In [ ]:
fig, tab = PR.fig_path_vs_setpoint(ALL, OUT, name="figP6c_path_vs_setpoint_catastrophic", catastrophic=True, drop_broken=True)
plt.show(); display(tab.round(1))

## Trap test for tracking DPC
The tracking controller trained through the TRUE plant, run through each surrogate. `ref_model_vs_true` ~ 1: the surrogate
judges a good controller correctly. `trap` = DPC's own controller / the reference controller, both on the surrogate: > 1 means
a better controller existed for this surrogate and DPC missed it.

In [ ]:
TD = PC.track_diagnostic(OUT, SUR_DIR, SEEDS, PLANT, rep=TRACK_REPS[0])
display(TD[TD.gap_true > 0.1].round(3))
display(TD.groupby("model")[["trap", "ref_model_vs_true", "gap_true"]].median().round(3))

## Figure P7 - Tracking DPC phase portraits: one figure per seed and setpoint

In [ ]:
C_TR = {m["name"]: PC.load_track_controllers(OUT, m["name"], TRACK_REPS[0]) for m in DC.MODELS if m.get("kind") != "true"}
TR = ["LN, MSE (3000)", "LN, J_x + J_u (3000)", "LN, two-stage", "static norm, MSE"]
for s in PHASE_SEEDS:
    for r in PHASE_REFS:
        PR.fig_phase_dpc([SURR[k] for k in TR], s, PLANT, OUT, {k: C_TR[k] for k in TR}, starts=STARTS, r=r, track=True,
                         name=f"figP7_track_dpc_seed{s}_r{r:g}"); plt.show()

## Save
Zips only this notebook's outputs (tables, tracking controllers and trajectories, figP* figures), not the whole DPC folder.

In [ ]:
import zipfile
z = OUT.rstrip("/") + "_phase_backup.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
    for root, _, files in os.walk(OUT):
        for n in files:
            p = os.path.join(root, n)
            rel = os.path.relpath(p, OUT)
            if (n.startswith(("phase_", "track_")) or rel.startswith(("track_", os.path.join("figs", "figP")))):
                f.write(p, rel)
print("zipped to", z, f"({os.path.getsize(z) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(z)
except ImportError:
    pass